# `searchFiles` — read-only file agent

```
query → route → recall → read → bundle → LLM → answer
```

Four tools, all read-only: `find` (names + folders, 4-tier similarity), `grep`
(contents), `read` (capped windows), `survey` (aggregate counts). Everything is
confined to `ROOTS` by `safe_resolve()`, which resolves *before* checking
containment, so `..` and escaping symlinks both fail.

Set a key with `export GROQ_API_KEY="gsk_..."`, then Restart Kernel → Run All.
`check_llm()` runs automatically and separates a wiring fault from an outage.

In [ ]:
# ===== config =====
from __future__ import annotations

import difflib
import fnmatch
import json
import os
import re
import subprocess
import time
from pathlib import Path

ROOTS = [Path("/home/princ3kr")]          # the jail

SKIP_DIRS = {
    ".git", "node_modules", ".venv", "venv", "__pycache__",
    ".mypy_cache", ".pytest_cache", ".ipynb_checkpoints", "dist", "build",
    ".vscode", "site-packages",
    ".cache", ".config", ".cargo", ".dotnet", ".nv", ".npm", ".snap", ".var",
    ".triton", ".gnupg", ".ipython", ".jupyter", ".icons", ".local",
    ".mozilla", ".thunderbird", "scikit_learn_data",
}

# Blocks read_file from decoding these as text. NOT a search filter -- a .zip is
# still findable and listable, it just cannot be read as text.
NO_TEXT_EXT = {
    ".pyc", ".so", ".dylib", ".dll", ".exe", ".o", ".a", ".bin",
    ".png", ".jpg", ".jpeg", ".gif", ".webp", ".ico", ".bmp", ".tiff",
    ".mp4", ".mov", ".mp3", ".wav", ".avi", ".mkv", ".webm",
    ".woff", ".woff2", ".ttf", ".eot", ".otf",
    ".pdf", ".docx", ".doc", ".xlsx", ".xls", ".pptx", ".ppt",
    ".zip", ".tar", ".gz", ".tgz", ".bz2", ".xz", ".7z", ".whl", ".rar", ".jar",
    ".db", ".sqlite", ".sqlite3", ".parquet", ".pkl", ".npy", ".npz",
    ".pt", ".pth", ".onnx", ".safetensors", ".ckpt", ".h5",
}
LISTABLE_ARCHIVE_EXT = {".zip": "zipfile", ".tar": "tarfile", ".gz": "tarfile",
                        ".tgz": "tarfile", ".bz2": "tarfile", ".xz": "tarfile"}

FIND_MAX_RESULTS  = 200
DIR_HINT_MAX      = 40
FUZZY_POOL_CAP    = 4_000
FUZZY_CUTOFF      = 0.60
# Fuzzy needs ENOUGH characters to mean anything. difflib scores short strings on
# incidental overlap: "search" vs "src" = 0.667 (just 's'+'rc'), "def" vs "ide" = 0.667.
# Below this length a ratio is noise, and it lets instruction prose like "Search the
# codebase" fuzzy-match every src/ folder on disk.
FUZZY_MIN_LEN     = 4
NEAR_MISS_MIN     = 0.15
NEAR_MISS_MAX     = 3
MAX_RESULTS       = 20
MAX_FILE_BYTES    = 50_000
MAX_LINES         = 200
GREP_MAX_PER_FILE = 3
READ_CONTEXT      = 25
BINARY_SNIFF      = 8_192
TIMEOUT_S         = 15
MAX_ARCHIVE_MEMBERS = 60

TOOLS = ("find", "grep", "read", "survey")
AGENT_NAME = "searchFiles"

## Logger

`list[dict]`, no database. Same fields the full harness will push to SQLite, so
swapping `LOGS.append()` for an insert should not touch a call site.

In [ ]:
# ===== logger =====
LOGS: list[dict] = []


def log(action: str, *, detail: str = "", status: str = "ok",
        t0: float | None = None, **extra) -> dict:
    rec = {"n": len(LOGS) + 1, "agent": AGENT_NAME, "action": action,
           "detail": detail, "status": status,
           "duration_ms": round((time.perf_counter() - t0) * 1000, 2) if t0 else None,
           "ts": time.strftime("%H:%M:%S")}
    rec.update(extra)
    LOGS.append(rec)
    return rec

## The jail — `safe_resolve`

The security model, enforced in **code, not in a prompt**.

1. **Containment** — resolved path must sit inside `ROOTS`. `.resolve()` runs first, so
   `..` traversal and escaping symlinks both fail.
2. **Read-only** — files open in `"rb"` only; no write, delete, or mutating shell-out.
3. **Allowlist** — `call_tool()` dispatches by name and fails closed on unknown names.

In [ ]:
# ===== jail + tool allowlist =====
ROOTS = [r.resolve() for r in ROOTS]


def safe_resolve(path) -> Path:
    p = Path(str(path)).expanduser()
    candidates = [p] if p.is_absolute() else [Path.cwd() / p] + [r / p for r in ROOTS]
    last_err = None
    for cand in candidates:
        try:
            rp = cand.resolve()
        except (OSError, RuntimeError) as e:
            last_err = e
            continue
        if any(rp == r or r in rp.parents for r in ROOTS):
            return rp
        last_err = None
    raise PermissionError(
        f"outside jail: {path!r} -> {p.absolute()} is not inside {[str(r) for r in ROOTS]}"
    ) from last_err


def call_tool(name: str, **kwargs):
    if name not in TOOLS:
        raise ValueError(f"tool {name!r} not allowed; allowed = {list(TOOLS)}")
    return {"find": find_files, "grep": search_content,
            "read": read_file, "survey": survey_files}[name](**kwargs)

## Tool — `find_files`

Searches **files and folders**. `find -type f` alone would miss `PracticeML/`, since no
*file* is named `practice`.

| rank | kind | `utils.py` matches |
|---|---|---|
| 0 | exact | `utils.py` |
| 1 | glob | `ut*.py` |
| 2 | substring | `utils_test.py`, `my_utils.py` |
| 3 | fuzzy ≥0.60 | `util.py` (0.89) |
| — | rejected | `unrelated.py` (0.38) |

Similarity is judged on the **stem**, and a stem match with a *different* extension is
tagged `stem_other_ext` so it can't masquerade as the file you asked for.

`near_misses()` fires when nothing matches. Its pool is restricted to the **same
extension** — extension agreement is a far stronger signal than string similarity, and
no ratio threshold separates `config/settings` (0.286, worth offering) from
`zzzzqqq/totally_unrelated` (0.269, pure filler-letter overlap).

In [ ]:
# ===== tool: find =====
def _stem(name: str) -> str:
    return Path(name).stem.lower()


def _match_kind(pattern: str, path):
    name = Path(path).name
    low, pl = name.lower(), pattern.lower()
    ps, ns = _stem(pl), _stem(low)
    if low == pl:
        return 0, 1.0, "exact"
    if fnmatch.fnmatch(low, pl):
        return 1, 0.95, "glob"
    same_ext = (Path(name).suffix.lower() == Path(pattern).suffix.lower()
                or not Path(pattern).suffix)
    if pl and (pl in low or (ps and ps in ns)):
        return (2, 0.80, "substring") if same_ext else (2, 0.70, "stem_other_ext")
    # short stems produce meaningless ratios -- see FUZZY_MIN_LEN
    if min(len(ns), len(ps)) < FUZZY_MIN_LEN:
        return None, 0.0, None
    ratio = difflib.SequenceMatcher(None, ns, ps).ratio() if (ns and ps) else 0.0
    if ratio >= FUZZY_CUTOFF:
        return 3, round(ratio, 3), "fuzzy"
    return None, round(ratio, 3), None


def _prune_args() -> list[str]:
    args = ["("]
    for i, d in enumerate(sorted(SKIP_DIRS)):
        if i:
            args.append("-o")
        args += ["-name", d]
    args += [")", "-prune", "-o"]
    return args


def _find_raw(pattern: str | None, base: Path, limit: int | None = None,
              kind: str = "f") -> list[str]:
    cmd = ["find", str(base), *_prune_args(), "-type", kind, "-print"]
    if pattern:
        # must precede -print or find ignores it and returns EVERY file
        cmd.insert(-1, "-iname")
        cmd.insert(-1, pattern)
    try:
        out = subprocess.run(cmd, capture_output=True, text=True,
                             timeout=TIMEOUT_S, check=False)
    except subprocess.TimeoutExpired:
        log("find", detail=f"{pattern or '*'} (TIMEOUT {TIMEOUT_S}s)",
            status="timeout", tool="find")
        return []
    lines = [l for l in out.stdout.splitlines() if l.strip()]
    return lines[:limit] if limit else lines


def _rank(pattern: str, raw: str, *, want_dirs: bool):
    try:
        rp = safe_resolve(raw)
    except PermissionError:
        return None
    is_dir = rp.is_dir()
    if is_dir != want_dirs:
        return None
    rank, score, kind = _match_kind(pattern, rp)
    if kind is None:
        return None
    return {"path": rp, "rank": rank, "kind": kind, "score": score, "is_dir": is_dir}


def find_files(pattern: str, root=None, max_results: int = FIND_MAX_RESULTS,
               fuzzy: bool = True, include_dirs: bool = True) -> list[dict]:
    t0 = time.perf_counter()
    base = safe_resolve(root or ROOTS[0])
    pattern_wild = pattern if any(ch in pattern for ch in "*?[") else f"*{pattern}*"

    best: dict[Path, dict] = {}

    def absorb(raws, want_dirs):
        for raw in raws:
            hit = _rank(pattern, raw, want_dirs=want_dirs)
            if hit and hit["path"] not in best:
                best[hit["path"]] = hit

    absorb(_find_raw(pattern_wild, base, kind="f"), False)
    if include_dirs:
        absorb(_find_raw(pattern_wild, base, kind="d"), True)

    if fuzzy and len(best) < max_results:
        ext = Path(pattern).suffix
        pool = f"*{ext}" if ext else None
        absorb(_find_raw(pool, base, limit=FUZZY_POOL_CAP + 1, kind="f"), False)
        if include_dirs:
            absorb(_find_raw(None, base, limit=FUZZY_POOL_CAP + 1, kind="d"), True)

    out = sorted(best.values(), key=lambda d: (d["rank"], -d["score"], str(d["path"])))
    out = out[:max_results]
    log("find", detail=f"{pattern} -> {len(out)} similar "
                      f"({sum(1 for d in out if d['is_dir'])} dir)",
        status="ok" if out else "not_found", t0=t0, found=len(out), tool="find")
    return out


def near_misses(pattern: str, root=None, limit: int = NEAR_MISS_MAX) -> list[dict]:
    t0 = time.perf_counter()
    base = safe_resolve(root or ROOTS[0])
    ext = Path(pattern).suffix
    pool = _find_raw(f"*{ext}" if ext else None, base, limit=FUZZY_POOL_CAP)
    ps = _stem(pattern)
    scored = []
    for raw in pool:
        try:
            rp = safe_resolve(raw)
        except PermissionError:
            continue
        ns = _stem(rp.name)
        if not (ns and ps) or ns == ps:
            continue
        ratio = difflib.SequenceMatcher(None, ns, ps).ratio()
        if ratio >= FUZZY_CUTOFF or ratio < NEAR_MISS_MIN:
            continue
        scored.append({"path": rp, "score": round(ratio, 3)})

    min_ratio = max(NEAR_MISS_MIN, 0.55) if len(pool) > 20 else NEAR_MISS_MIN
    scored = [s for s in scored if s["score"] >= min_ratio]
    scored.sort(key=lambda d: -d["score"])
    out = scored[:limit]
    log("near_miss", detail=f"{pattern} -> {len(out)}", status="ok" if out else "none",
        t0=t0, tool="near_miss")
    return out

In [ ]:
# ===== tool: grep =====
def search_content(query: str, root=None, max_results: int = MAX_RESULTS,
                   regex: bool = False,
                   max_per_file: int = GREP_MAX_PER_FILE) -> list[dict]:
    t0 = time.perf_counter()
    base = safe_resolve(root or ROOTS[0])

    cmd = ["grep", "-rI", "-n", "-m", str(max_per_file)]
    cmd += ["-E" if regex else "-F", query]
    for d in sorted(SKIP_DIRS):
        cmd.append(f"--exclude-dir={d}")
    for e in sorted(NO_TEXT_EXT):
        cmd.append(f"--exclude=*{e}")
    cmd.append(str(base))

    try:
        out = subprocess.run(cmd, capture_output=True, text=True,
                             timeout=TIMEOUT_S, check=False)
    except subprocess.TimeoutExpired:
        log("grep", detail=f"{query!r} (TIMEOUT {TIMEOUT_S}s)", status="timeout", t0=t0)
        return []

    hits, seen = [], {}
    for line in out.stdout.splitlines():
        if len(hits) >= max_results:
            break
        if ":" not in line:
            continue
        raw_path, _, rest = line.partition(":")
        parts = rest.split(":", 1)
        if len(parts) != 2 or not parts[0].isdigit():
            continue
        line_no, body = int(parts[0]), parts[1]
        if seen.get(raw_path, 0) >= max_per_file:
            continue
        try:
            p = safe_resolve(raw_path)
        except PermissionError:
            continue
        seen[raw_path] = seen.get(raw_path, 0) + 1
        hits.append({"path": p, "line_no": line_no, "text": body.strip()[:300]})

    log("grep", detail=f"{query!r} -> {len(hits)} hit(s)",
        status="ok" if hits else "not_found", t0=t0, found=len(hits), tool="grep")
    return hits

## Tool — `read_file`

**Findable ≠ readable.** `find_files` applies no extension filter; `read_file` decides
what is decodable and says why when it isn't. Archives are **listed, never extracted**
(`zipfile`/`tarfile` are stdlib, so no install). Binary content is never returned.

In [ ]:
# ===== tool: read =====
def _list_archive(p: Path, t0: float) -> dict:
    import zipfile
    import tarfile

    ext = p.suffix.lower()
    try:
        if ext == ".zip":
            with zipfile.ZipFile(p) as z:
                names = [f"{i.filename}  ({i.file_size:,}B)" for i in z.infolist()
                         if not i.is_dir()]
        else:
            mode = "r:gz" if ext == ".gz" else ("r:bz2" if ext == ".bz2" else
                                                 ("r:xz" if ext == ".xz" else "r"))
            with tarfile.open(p, mode) as t:
                names = [f"{m.name}  ({m.size:,}B)" for m in t.getmembers() if m.isfile()]
    except (zipfile.BadZipFile, tarfile.TarError, OSError, EOFError) as e:
        log("read", detail=f"{p.name} ({type(e).__name__})", status="error", t0=t0)
        return {"path": str(p), "status": "error",
                "reason": f"could not list archive: {type(e).__name__}: {e}", "text": ""}

    truncated = len(names) > MAX_ARCHIVE_MEMBERS
    body = "\n".join(f"{i:>4} | {n}" for i, n in enumerate(names[:MAX_ARCHIVE_MEMBERS], 1))
    text = (f"ARCHIVE: {p.name}  ({len(names)} file(s), "
            f"{p.stat().st_size:,}B compressed){'  [TRUNCATED]' if truncated else ''}\n" + body)

    log("read", detail=f"{p.name} -> {len(names)} member(s)", status="ok", t0=t0)
    return {"path": str(p), "status": "ok", "start_line": 1,
            "n_lines": min(len(names), MAX_ARCHIVE_MEMBERS), "total_lines": len(names),
            "truncated": truncated, "text": text,
            "meta": {"kind": "archive_listing", "members": len(names)}}


def read_file(path, start_line: int = 1, max_lines: int = MAX_LINES,
              max_bytes: int = MAX_FILE_BYTES) -> dict:
    t0 = time.perf_counter()
    try:
        p = safe_resolve(path)
    except PermissionError as e:
        log("read", detail=str(path), status="blocked", t0=t0)
        return {"path": str(path), "status": "blocked", "reason": str(e), "text": ""}

    if not p.is_file():
        log("read", detail=str(p), status="not_found", t0=t0)
        return {"path": str(p), "status": "not_found", "reason": "not a file", "text": ""}

    ext = p.suffix.lower()
    try:
        size = p.stat().st_size
    except OSError as e:
        return {"path": str(p), "status": "error", "reason": str(e), "text": ""}

    if ext in LISTABLE_ARCHIVE_EXT:
        return _list_archive(p, t0)

    if ext in NO_TEXT_EXT:
        log("read", detail=f"{p.name} ({ext})", status="skipped", t0=t0)
        hint = {".pdf": "install pypdf", ".docx": "install python-docx",
                ".doc": "install antiword/libreoffice",
                ".xlsx": "install openpyxl"}.get(ext, f"{ext} is binary")
        return {"path": str(p), "status": "skipped", "reason": hint, "text": "",
                "meta": {"size_bytes": size, "ext": ext}}

    if size > max_bytes * 4:
        log("read", detail=f"{p.name} {size}B > cap", status="skipped", t0=t0)
        return {"path": str(p), "status": "skipped",
                "reason": f"file is {size:,}B, too large to read blindly", "text": "",
                "meta": {"size_bytes": size, "ext": ext}}

    try:
        with open(p, "rb") as fh:
            head = fh.read(BINARY_SNIFF)
            if b"\x00" in head:
                log("read", detail=f"{p.name} (binary)", status="binary", t0=t0)
                return {"path": str(p), "status": "binary",
                        "reason": "contains NUL bytes", "text": ""}
            fh.seek(0)
            raw = fh.read(max_bytes)
    except OSError as e:
        return {"path": str(p), "status": "error", "reason": str(e), "text": ""}

    lines = raw.decode("utf-8", errors="replace").splitlines()
    start = max(1, int(start_line))
    window = lines[start - 1: start - 1 + max_lines]
    shown = "\n".join(f"{start + i:>5} | {ln}" for i, ln in enumerate(window))

    log("read", detail=f"{p.name} L{start}-L{start + len(window) - 1}",
        status="ok", t0=t0, lines=len(window))
    return {"path": str(p), "status": "ok", "start_line": start,
            "n_lines": len(window), "total_lines": len(lines),
            "truncated": len(window) < (len(lines) - start + 1), "text": shown}

## Tool — `survey_files`

Aggregate counts. Reading files cannot answer *"how many images"*; that needs counting.
One walk returns per-extension, per-category, and per-top-level distributions.

**Counts are only meaningful with their scope.** Unpruned `find` reports 1143 images;
with `SKIP_DIRS` applied, 51. Both true, wildly different — so the bundle always lists
the exclusions and the system prompt requires stating them.

In [ ]:
# ===== tool: survey =====
CATEGORIES = {
    "image": {".png", ".jpg", ".jpeg", ".gif", ".webp", ".bmp", ".ico", ".tiff",
              ".svg", ".heic", ".avif"},
    "video": {".mp4", ".mkv", ".mov", ".avi", ".webm", ".flv", ".wmv", ".m4v"},
    "audio": {".mp3", ".wav", ".flac", ".ogg", ".m4a", ".aac", ".wma"},
    "document": {".pdf", ".doc", ".docx", ".odt", ".rtf", ".txt", ".md", ".tex",
                 ".epub", ".ppt", ".pptx", ".xls", ".xlsx"},
    "spreadsheet": {".csv", ".tsv", ".xls", ".xlsx", ".ods", ".parquet"},
    "notebook": {".ipynb"},
    "archive": {".zip", ".tar", ".gz", ".tgz", ".bz2", ".xz", ".7z", ".rar", ".jar"},
    "code": {".py", ".js", ".jsx", ".ts", ".tsx", ".java", ".c", ".h", ".cpp",
             ".hpp", ".cs", ".go", ".rs", ".rb", ".php", ".pl", ".lua", ".r",
             ".jl", ".swift", ".kt", ".scala", ".hs", ".ml", ".ex", ".exs",
             ".sh", ".bash", ".zsh", ".sql", ".html", ".css", ".vue", ".svelte"},
    "data": {".json", ".jsonl", ".yaml", ".yml", ".toml", ".ini", ".cfg", ".xml"},
    "binary": {".so", ".dll", ".dylib", ".exe", ".bin", ".o", ".a", ".pyc", ".wasm"},
}


def survey_files(root=None, categories=None, extensions=None) -> dict:
    t0 = time.perf_counter()
    base = safe_resolve(root or ROOTS[0])
    cats = ([categories] if isinstance(categories, str) else list(categories or CATEGORIES))
    wanted: set[str] = set()
    for c in cats:
        wanted |= CATEGORIES.get(c, set())
    if extensions:
        wanted |= {e.lower() if e.startswith(".") else f".{e.lower()}" for e in extensions}

    by_ext: dict[str, int] = {}
    by_cat: dict[str, int] = {c: 0 for c in cats}
    by_top: dict[str, int] = {}
    sample: list[str] = []
    seen = 0

    for raw in _find_raw(None, base, kind="f"):
        seen += 1
        p = Path(raw)
        ext = p.suffix.lower()
        by_ext[ext or "(none)"] = by_ext.get(ext or "(none)", 0) + 1
        rel = p.relative_to(base).parts
        top = rel[0] if rel else "(root)"
        by_top[top] = by_top.get(top, 0) + 1
        if ext in wanted:
            for c in cats:
                if ext in CATEGORIES.get(c, set()):
                    by_cat[c] += 1
            if len(sample) < 200:
                sample.append(raw)

    log("survey", detail=f"{seen} files, cats={cats}", status="ok", t0=t0)
    return {"root": str(base), "files_seen": seen,
            "by_extension": dict(sorted(by_ext.items(), key=lambda kv: -kv[1])[:25]),
            "by_category": by_cat,
            "by_toplevel": dict(sorted(by_top.items(), key=lambda kv: -kv[1])[:15]),
            "matched_count": sum(by_cat.values()), "sample": sample[:12],
            "excluded_dirs": sorted(SKIP_DIRS),
            "note": "Counts EXCLUDE the pruned directories. Ask about a specific "
                    "folder to count within it."}

## The agent

```
intent → route → recall → read → evidence
```

`detect_query_type()` splits **aggregate** questions (count extensions) from **locate**
questions (match names). Conflating them is why "how many images" once matched folders
literally named `images` and reported a confidently wrong answer.

This router is a pure function of the query string — the exact seam a DSPy module would
replace. Every decision is logged with `action='route'`, and **that log is the training
set**. Note the router is *deterministic on purpose*: an unoptimised model is worse than
regex when you have no labelled examples.

Known gaps: no global ranking when several files are named at once; exhaustive name
search does not additionally grep symbols inside; `search_content` has no size guard
(this `grep` lacks `--max-filesize`).

In [ ]:
# ===== agent loop =====
CATEGORY_WORDS = {
    "python": "code", "py": "code", "javascript": "code", "js": "code",
    "typescript": "code", "ts": "code", "java": "code", "rust": "code",
    "golang": "code", "cpp": "code", "csharp": "code", "ruby": "code",
    "image": "image", "images": "image", "picture": "image", "pictures": "image",
    "photo": "image", "photos": "image", "screenshot": "image", "icon": "image",
    "icons": "image", "video": "video", "videos": "video", "audio": "audio",
    "music": "audio", "song": "audio", "songs": "audio",
    "document": "document", "documents": "document", "doc": "document",
    "docs": "document", "pdf": "document", "pdfs": "document", "note": "document",
    "notes": "document", "text": "document",
    "spreadsheet": "spreadsheet", "spreadsheets": "spreadsheet", "csv": "spreadsheet",
    "notebook": "notebook", "notebooks": "notebook",
    "archive": "archive", "archives": "archive", "zip": "archive", "zips": "archive",
    "code": "code", "codes": "code", "script": "code", "scripts": "code",
    "source": "code", "file": None, "files": None,
    "data": "data", "dataset": "data", "datasets": "data",
    "binary": "binary", "binaries": "binary", "executable": "binary",
}

AGGREGATE_CUES = (
    "how many", "how much", "count", "number of", "total", "tally",
    "what type", "what types", "what kind", "what kinds", "what format",
    "which type", "which types", "what extension", "what extensions",
    "breakdown", "summary", "summarize", "summarise", "overview",
    "biggest", "largest", "smallest", "oldest", "newest",
    "list all", "show all", "statistics", "stats",
)

# Must cover archives and documents: when `zip` was missing,
# "what is inside Ask-My-Repo-mainV2.zip?" captured no filename at all.
FILENAME_EXTS = (
    "py|ipynb|md|txt|json|jsonl|ya?ml|toml|cfg|ini|env|lock|csv|tsv|log|sql|db|sqlite"
    "|js|jsx|mjs|cjs|ts|tsx|vue|svelte|html|htm|css|scss|sh|bash|zsh|fish|ps1|bat"
    "|c|h|cc|cpp|hpp|cxx|cs|java|kt|kts|go|rs|rb|php|pl|lua|r|jl|swift|dart|ex|exs"
    "|scala|clj|hs|ml|mli|erl|zig|nim|vim|el|asm|s"
    "|zip|tar|gz|tgz|bz2|xz|7z|rar|jar|whl"
    "|pdf|docx?|xlsx?|pptx?|epub|rtf|odt|ods"
    "|png|jpe?g|gif|webp|bmp|tiff?|svg|ico|mp4|mov|avi|mkv|webm|mp3|wav|flac"
)
FILENAME_RE = re.compile(r"[\w\-./~]*[\w\-]+\.(?:" + FILENAME_EXTS + r")\b", re.I)
BARE_FILENAME_RE = re.compile(
    r"\b(?:Dockerfile|Makefile|README|LICENSE|requirements|Procfile)\b", re.I)

STOPWORDS = {
    "the", "a", "an", "is", "are", "was", "in", "on", "at", "to", "of", "and", "or",
    "for", "with", "why", "how", "what", "where", "which", "does", "do", "did", "can",
    "you", "i", "me", "my", "it", "this", "that", "there", "here", "be", "been",
    "not", "but", "if", "then", "so", "about", "from", "by", "please", "help",
    "show", "find", "tell", "explain", "look", "check", "file", "files", "code",
    "use", "using", "get",
    # instruction verbs/nouns. Without these, a planner instruction like
    # "Search the codebase for the definition of X" leaks `search` and `def` as
    # keywords, and a folder-hint lookup then matches src/ and ide/.
    "search", "searching", "locate", "identify", "definition", "defined",
    "defines", "codebase", "repository", "repo", "project", "folder",
    "directory", "list", "give", "return", "find", "report", "summarise",
    "count", "total", "number", "e.g", "eg", "example", "examples", "etc",
    "return", "contains", "containing", "across", "within",
}


def detect_query_type(query: str) -> dict:
    q = query.lower()
    words = set(re.findall(r"[a-z]+", q))
    is_agg = any(cue in q for cue in AGGREGATE_CUES)
    cats = sorted({CATEGORY_WORDS[w] for w in words
                   if w in CATEGORY_WORDS and CATEGORY_WORDS[w]})
    return {"shape": "aggregate" if is_agg else "locate", "aggregate": is_agg,
            "categories": cats,
            "cue_hit": next((c for c in AGGREGATE_CUES if c in q), None)}


def extract_intent(query: str) -> dict:
    filenames, seen = [], set()
    for pat in (FILENAME_RE, BARE_FILENAME_RE):
        for m in pat.finditer(query):
            fn = m.group(0).strip("./")
            if fn.lower() not in seen:
                seen.add(fn.lower())
                filenames.append(fn)
    keywords = [w for w in re.findall(r"[A-Za-z_][A-Za-z0-9_.]{2,}", query)
                if w.lower() not in STOPWORDS and not w.endswith(".")]
    # drop fragments of captured filenames, or "Ask" from "Ask-My-Repo.zip"
    # triggers a junk folder search
    fn_words = {w.lower() for f in filenames for w in re.split(r"[.\-_/]", f) if w}
    keywords = [w for w in keywords if w.lower() not in fn_words]
    return {"filenames": filenames, "keywords": list(dict.fromkeys(keywords))[:5],
            "raw": query}


def _describe_dir(d: Path, max_children: int = 40) -> dict:
    t0 = time.perf_counter()
    try:
        kids_dirs = [c.name + "/" for c in sorted(d.iterdir())
                     if c.is_dir() and c.name not in SKIP_DIRS]
        kids_files = [c.name for c in sorted(d.iterdir()) if c.is_file()]
    except (OSError, PermissionError) as e:
        return {"path": str(d), "kind": "folder", "is_dir": True, "score": 0.0,
                "read_status": "error", "truncated": False,
                "excerpt": f"(could not list: {e})"}

    # histogram over the whole subtree; counting children AND rglob double-counts
    counts: dict[str, int] = {}
    sub_files = sub_dirs = 0
    try:
        for p in d.rglob("*"):
            if p.is_dir():
                sub_dirs += 1
            elif p.is_file():
                sub_files += 1
                e = p.suffix.lower() or "(no ext)"
                counts[e] = counts.get(e, 0) + 1
    except OSError:
        pass
    top_ext = sorted(counts.items(), key=lambda kv: -kv[1])[:6]

    lines = [f"FOLDER: {d}",
             f"  child folders ({len(kids_dirs)}): "
             + (", ".join(kids_dirs[:max_children]) or "(none)"),
             "  types in tree: "
             + (", ".join(f"{e} x{n}" for e, n in top_ext) or "(none)")]
    if kids_files:
        lines.append("  files here: " + ", ".join(kids_files[:20]))
    if sub_dirs or sub_files:
        lines.append(f"  recursively: {sub_dirs} subfolder(s), "
                     f"{sub_files} file(s) in total")

    log("read", detail=f"{d.name}/ ({len(kids_dirs)}d {len(kids_files)}f)",
        status="ok", t0=t0)
    return {"path": str(d), "kind": "folder", "is_dir": True, "score": 0.0,
            "read_status": "ok", "truncated": len(kids_dirs) > max_children,
            "excerpt": "\n".join(lines)}


def run_agent(user_query: str, top_files: int = 3, root=None,
              exhaustive: bool | None = None) -> dict:
    t_start = time.perf_counter()
    intent = extract_intent(user_query)
    qtype = detect_query_type(user_query)
    intent["query_type"] = qtype
    if exhaustive is None:
        exhaustive = bool(intent["filenames"])

    _rd = qtype["shape"] + (f" cats={qtype['categories']}" if qtype["categories"] else "")
    log("route", detail=_rd, status="ok", shape=qtype["shape"],
        categories=qtype["categories"], cue=qtype["cue_hit"], tool="router")

    # ---- aggregate: "how many images are there?"
    if qtype["aggregate"]:
        cats = qtype["categories"]
        sv = call_tool("survey", root=root, categories=cats or None)
        rec = log("run_agent", detail=user_query[:60], status="ok", t0=t_start,
                  shape="aggregate", cats=cats, tool="agent")
        ok = bool(cats) or bool(intent["filenames"])
        return {"status": "ok", "query": user_query, "intent": intent,
                "route": "aggregate", "near_misses": [],
                "searches": [{"by": "survey", "categories": cats}],
                "survey": sv, "evidence": [], "similar_files": [],
                "summary": (f"Surveyed {sv['files_seen']} files; "
                            f"{sv['matched_count']} in {cats}." if ok else
                            f"Surveyed {sv['files_seen']} files, no category recognised."),
                "confidence": 0.85 if ok else 0.4, "elapsed_ms": rec["duration_ms"]}

    ranked: list[dict] = []
    seen_paths: set[Path] = set()
    searches: list[dict] = []

    # 1. name recall: ALL filename candidates, union, no early break
    for fn in intent["filenames"]:
        found = call_tool("find", pattern=fn, root=root, max_results=FIND_MAX_RESULTS)
        searches.append({"by": "find", "query": fn, "n": len(found),
                         "folders": sum(1 for f in found if f["is_dir"])})
        for h in found:
            if h["path"] not in seen_paths:
                seen_paths.add(h["path"])
                ranked.append(h)

    # 1b. bare keywords have no extension, so FILENAME_RE misses them:
    #     "show me the rust project" needs a folder lookup.
    #     fuzzy=False on purpose -- a folder hint must be a real name match, else
    #     instruction prose ("Search the codebase") matches src/ everywhere.
    for kw in intent["keywords"]:
        if len(kw) < 3 or any(kw.lower() == f.lower() for f in intent["filenames"]):
            continue
        hits = [f for f in call_tool("find", pattern=kw, root=root, fuzzy=False,
                                     max_results=DIR_HINT_MAX) if f["is_dir"]]
        if hits:
            searches.append({"by": "find_dir", "query": kw, "n": len(hits)})
        for h in hits:
            if h["path"] not in seen_paths:
                seen_paths.add(h["path"])
                ranked.append(h)

    # 2. content recall: only when no name matched, or when forced exhaustive
    if not ranked:
        for kw in (intent["keywords"] if exhaustive else intent["keywords"][:1]):
            found = call_tool("grep", query=kw, root=root, max_results=MAX_RESULTS)
            searches.append({"by": "grep", "query": kw, "n": len(found)})
            for h in found:
                if h["path"] not in seen_paths:
                    seen_paths.add(h["path"])
                    ranked.append({"path": h["path"], "rank": 9, "kind": "content",
                                   "score": 0.0, "is_dir": False,
                                   "line_no": h["line_no"], "text": h["text"]})

    if not ranked:
        # offer near misses before giving up
        nm = []
        for fn in (intent["filenames"] or intent["keywords"][:2]):
            nm.extend(near_misses(fn, root=root))
        uniq, s = [], set()
        for c in nm:
            if c["path"] not in s:
                s.add(c["path"])
                uniq.append(c)
        uniq = uniq[:NEAR_MISS_MAX]
        rec = log("run_agent", detail=user_query[:60],
                  status="near_miss" if uniq else "not_found", t0=t_start,
                  near=len(uniq), tool="agent")
        return {"status": "near_miss" if uniq else "not_found",
                "query": user_query, "intent": intent, "searches": searches,
                "evidence": [], "similar_files": [], "near_misses": uniq,
                "summary": (f"No exact match, {len(uniq)} similar name(s) exist."
                            if uniq else "No file matched."),
                "confidence": 0.3 if uniq else 0.0, "elapsed_ms": rec["duration_ms"]}

    # 3. read top N files, centred on the match; folders get summarised
    evidence = []
    for h in ranked:
        if len([e for e in evidence if e["kind"] != "folder"]) >= top_files:
            break
        if h.get("is_dir"):
            evidence.append(_describe_dir(h["path"]))
            continue
        line = h.get("line_no")
        r = call_tool("read", path=h["path"],
                      start_line=(line - READ_CONTEXT if line else 1))
        evidence.append({"path": str(h["path"]), "kind": h["kind"], "score": h["score"],
                         "is_dir": False, "match_line": line,
                         "match_text": h.get("text"), "read_status": r["status"],
                         "truncated": r.get("truncated"), "excerpt": r.get("text", "")[:6000]})

    similar = [{"path": str(h["path"]), "kind": h["kind"], "score": h["score"],
                "is_dir": bool(h.get("is_dir"))} for h in ranked]
    by_grep = any(s["by"] == "grep" for s in searches)
    rec = log("run_agent", detail=user_query[:60], status="ok", t0=t_start,
              files=len(evidence), similar=len(similar), tool="agent")
    return {"status": "ok", "query": user_query, "intent": intent, "route": "locate",
            "searches": searches, "evidence": evidence, "similar_files": similar,
            "near_misses": [],
            "summary": f"Found {len(similar)} similar file(s); read {len(evidence)}.",
            "confidence": 0.8 if not by_grep else 0.6, "elapsed_ms": rec["duration_ms"]}

## Prompt bundle

Compact and citable — paths plus line numbers, never a raw dump. Three shapes:

- **aggregate** — counts *plus the exclusions*, so the number is interpretable
- **near_miss** — "no exact match, did you mean…", never a near miss as a match
- **locate** — excerpts with `path:line` citations

In [ ]:
# ===== prompt bundle =====
def build_prompt_bundle(result: dict) -> str:
    if result.get("route") == "aggregate":
        sv = result["survey"]
        cats = result["searches"][0]["categories"]
        out = [f"AGENT: {AGENT_NAME} (read-only)  [AGGREGATE QUERY]",
               f"QUERY: {result['query']}", f"FILES SURVEYED: {sv['files_seen']}",
               f"ROOT: {sv['root']}", ""]
        if cats:
            out.append(f"COUNT BY CATEGORY ({', '.join(cats)}):")
            for c, n in sorted(sv["by_category"].items(), key=lambda kv: -kv[1]):
                out.append(f"    {c:<12} {n}")
            out += [f"    {'TOTAL':<12} {sv['matched_count']}", "",
                    "EXTENSION BREAKDOWN (top 15):"]
            out += [f"    {e:<12} {n}" for e, n in list(sv["by_extension"].items())[:15]]
            if sv["sample"]:
                out += ["", "SAMPLE MATCHES (proof the count is real):"]
                out += [f"    {s}" for s in sv["sample"][:6]]
        else:
            out.append("No category word recognised. Top-level distribution:")
            out += [f"    {k:<22} {v}" for k, v in list(sv["by_toplevel"].items())[:12]]
        out += ["", "=" * 60, "COUNTING CAVEAT (must be stated in the answer):",
                f"  These counts EXCLUDE: {', '.join(sv['excluded_dirs'])}",
                "  This is the count of files the agent can see, NOT the whole disk.",
                f"CONFIDENCE: {result['confidence']}   ({result['elapsed_ms']}ms)",
                "INSTRUCTION: Answer with the exact number. State the exclusions. "
                "Do NOT re-count from the sample."]
        return "\n".join(out)

    if result["status"] == "near_miss":
        out = [f"AGENT: {AGENT_NAME} (read-only)  [NO EXACT MATCH]",
               f"QUERY: {result['query']}",
               f"SEARCHES ATTEMPTED: {result['searches']}", "",
               f"NEAR MISSES ({len(result['near_misses'])}):"]
        out += [f"    score {c['score']:.2f}  {c['path']}"
                for c in result["near_misses"]]
        out += ["=" * 60,
                "INSTRUCTION: No exact match exists. Say so plainly, then offer the "
                "closest matches and ask which was meant. Do NOT present a near miss "
                "as if it were the requested file."]
        return "\n".join(out)

    if result["status"] == "not_found":
        return (f"AGENT: {AGENT_NAME}\nQUERY: {result['query']}\nRESULT: not_found\n"
                f"SEARCHES ATTEMPTED: {result['searches']}\n"
                f"INSTRUCTION: Tell the user plainly that nothing matched. "
                f"Do NOT guess a filename or invent content.")

    sim = result.get("similar_files") or []
    out = [f"AGENT: {AGENT_NAME} (read-only)", f"QUERY: {result['query']}",
           f"SEARCHES: {result['searches']}"]
    if len(sim) > len(result["evidence"]):
        out.append(f"ALL {len(sim)} SIMILAR FILES FOUND (only the first "
                   f"{len(result['evidence'])} were read):")
        out += [f"    [{s['kind']:<9} {s['score']}] {s['path']}" for s in sim]

    out += ["", f"EVIDENCE ({len(result['evidence'])} item(s) read):", ""]
    for i, e in enumerate(result["evidence"], 1):
        label = "folder" if e.get("is_dir") else e["kind"]
        out.append(f"--- [{i}] {e['path']}   (matched by {label})")
        if e.get("is_dir"):
            out += ["    (folder listing, not file contents)", e["excerpt"], ""]
            continue
        if e["match_line"]:
            out.append(f"    line {e['match_line']}: {e['match_text']}")
        if e["read_status"] != "ok":
            out.append(f"    (not read: {e['read_status']})")
            continue
        if e["truncated"]:
            out.append("    (excerpt, truncated)")
        out += [e["excerpt"], ""]

    out += ["=" * 60, f"CONFIDENCE: {result['confidence']}   ({result['elapsed_ms']}ms)",
            "INSTRUCTION: Answer the query in natural language using ONLY the evidence "
            "above. Cite files as path:line. If the evidence does not answer it, say so "
            "plainly rather than guessing."]
    return "\n".join(out)

## LLM

Any OpenAI-compatible `/chat/completions` — Groq, OpenAI, OpenRouter, or local Ollama.

```bash
export GROQ_API_KEY="gsk_..."
export LLM_MODEL="openai/gpt-oss-120b"   # optional
```

Verified on Groq: `openai/gpt-oss-120b`, `openai/gpt-oss-20b`, `qwen/qwen3.8-27b`.
An inaccessible model returns **404, not 401**, which looks like a typo rather than a
permissions problem — hence the preflight.

**Config faults are loud, transient faults are quiet.** A rejected key or unknown model
raises `LLMConfigError` and prints the exact `export` commands; only timeouts/5xx/429
degrade quietly to the raw bundle. Conflating these once made a 404 model report as
"no API key", which sent debugging in the wrong direction.

`gpt-oss-*` are reasoning models: they spend 35–50 tokens on hidden reasoning, and a
budget below that returns empty content. `call_llm` raises rather than returning "".

In [ ]:
# ===== LLM client =====
import requests

LLM_BASE_URL = os.getenv("LLM_BASE_URL")
LLM_MODEL    = os.getenv("LLM_MODEL")
LLM_TIMEOUT  = 60
LLM_TEMPERATURE = 0.0

# Inline fallback so the notebook runs without exporting anything.
# !! Stored in PLAINTEXT -- prefer the env var and blank this out.
_INLINE_KEY = os.getenv("LLM_API_KEY")

# Other providers (OpenAI-compatible, so just swap these two):
#   LLM_BASE_URL, LLM_MODEL = "https://api.openai.com/v1", "gpt-4o-mini"
#   LLM_BASE_URL, LLM_MODEL = "https://openrouter.ai/api/v1", "anthropic/claude-3.5-sonnet"
#   LLM_BASE_URL, LLM_MODEL = "http://localhost:11434/v1", "qwen3.2"   # Ollama, no key

GROQ_CHAT_MODELS = ["openai/gpt-oss-120b", "openai/gpt-oss-20b", "qwen/qwen3.8-27b"]

SYSTEM_PROMPT = """You are a precise assistant answering questions about files on a user's computer.

Rules:
- Use ONLY the evidence provided. Never invent file names, counts, line numbers, or code.
- Cite sources inline as path/to/file.ext:12.

For AGGREGATE queries (counts, totals, breakdowns):
- Give the exact number from the COUNT BY CATEGORY section. Do not estimate.
- ALWAYS state that the count excludes the pruned directories, and name a few.
  "50 images" and "1143 images" are both true for different scopes; say which you mean.
- Do not count the sample lines to get a number.

For NO EXACT MATCH:
- Say clearly that no exact match exists.
- Then offer the near misses as possibilities and ask which was intended.
- Never present a near miss as if it were the requested file.

For LOCATE queries:
- If the evidence does not answer it, say so plainly and name what is missing.
- If a file could not be read, say so rather than guessing its contents.
- When several similar files were found, list them and explain the difference.

Be concise and concrete. No preamble, no restating the question."""


class LLMUnavailable(RuntimeError):
    """Transient: timeout, 429, 5xx. Safe to degrade."""


class LLMConfigError(LLMUnavailable):
    """Config bug: bad key, unknown model. Must be reported loudly."""


def api_key() -> str:
    """Read at CALL time so exporting the key later needs no cell re-run."""
    return os.getenv("GROQ_API_KEY") or os.getenv("LLM_API_KEY") or _INLINE_KEY or ""


def llm_status() -> dict:
    k = api_key()
    return {"configured": bool(k), "base_url": LLM_BASE_URL, "model": LLM_MODEL,
            "key_source": ("GROQ_API_KEY env" if os.getenv("GROQ_API_KEY")
                           else "LLM_API_KEY env" if os.getenv("LLM_API_KEY")
                           else "inline in notebook" if k else "none")}


def build_llm_messages(result: dict) -> list[dict]:
    return [{"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": build_prompt_bundle(result)}]


def call_llm(messages: list[dict], *, temperature: float = LLM_TEMPERATURE,
             max_tokens: int = 900) -> dict:
    key = api_key()
    if not key:
        raise LLMConfigError("no API key: export GROQ_API_KEY or fill _INLINE_KEY")

    t0 = time.perf_counter()
    payload = {"model": LLM_MODEL, "messages": messages,
               "temperature": temperature, "max_tokens": max_tokens}
    headers = {"Authorization": f"Bearer {key}", "Content-Type": "application/json"}

    try:
        r = requests.post(f"{LLM_BASE_URL}/chat/completions", headers=headers,
                          json=payload, timeout=LLM_TIMEOUT)
    except requests.Timeout:
        log("llm", detail=f"{LLM_MODEL} timeout", status="error", t0=t0, tool="llm")
        raise LLMUnavailable(f"timeout after {LLM_TIMEOUT}s") from None
    except requests.RequestException as e:
        log("llm", detail=type(e).__name__, status="error", t0=t0, tool="llm")
        raise LLMUnavailable(f"network error: {type(e).__name__}: {e}") from e

    if r.status_code in (401, 403):
        log("llm", detail=f"HTTP {r.status_code} auth", status="config_error", t0=t0)
        raise LLMConfigError(f"HTTP {r.status_code}: key rejected by {LLM_BASE_URL}.")
    if r.status_code == 404:
        log("llm", detail=f"HTTP 404 model {LLM_MODEL}", status="config_error", t0=t0)
        raise LLMConfigError(
            f"HTTP 404: model {LLM_MODEL!r} not found or not accessible. "
            f"Known-good on Groq: {', '.join(GROQ_CHAT_MODELS)}.")
    if r.status_code == 429:
        log("llm", detail="HTTP 429", status="error", t0=t0)
        raise LLMUnavailable("rate limited (HTTP 429) -- retry shortly")
    if r.status_code >= 500:
        log("llm", detail=f"HTTP {r.status_code}", status="error", t0=t0)
        raise LLMUnavailable(f"server error HTTP {r.status_code}")
    if r.status_code != 200:
        log("llm", detail=f"HTTP {r.status_code}", status="error", t0=t0)
        raise LLMUnavailable(f"HTTP {r.status_code}: {r.text[:200]}")

    data = r.json()
    try:
        text = data["choices"][0]["message"]["content"].strip()
    except (KeyError, IndexError, TypeError) as e:
        log("llm", detail="malformed response", status="error", t0=t0)
        raise LLMUnavailable(f"unexpected shape: {str(data)[:200]}") from e

    usage = data.get("usage", {}) or {}
    reasoning = (usage.get("completion_tokens_details") or {}).get("reasoning_tokens")
    if not text:
        # reasoning models can burn the whole budget on hidden reasoning
        log("llm", detail="empty content (max_tokens too small?)", status="error", t0=t0)
        raise LLMUnavailable(
            f"empty response from {LLM_MODEL} at max_tokens={max_tokens}"
            + (f" (reasoning_tokens={reasoning})" if reasoning else "")
            + " -- raise max_tokens, or use a non-reasoning model.")

    log("llm", detail=f"{data.get('model', LLM_MODEL)} -> {len(text)} chars",
        status="ok", t0=t0, tokens=usage.get("total_tokens"), tool="llm")
    return {"text": text, "usage": usage, "model": data.get("model", LLM_MODEL)}


def check_llm() -> bool:
    """Preflight: separates 'wired wrong' from 'service down'."""
    st = llm_status()
    print(f"  key    : {'SET' if st['configured'] else 'NOT SET'}  ({st['key_source']})")
    print(f"  model  : {st['model']}")
    print(f"  base   : {st['base_url']}")
    if not st["configured"]:
        print("  RESULT : FAIL - no key")
        return False
    t0 = time.perf_counter()
    try:
        out = call_llm([{"role": "user", "content": "Reply with exactly: PONG"}],
                       max_tokens=256)
        print(f"  RESULT : PASS - {out['model']} replied {out['text'][:20]!r} "
              f"in {round((time.perf_counter()-t0)*1000)}ms")
        return True
    except LLMConfigError as e:
        print(f"  RESULT : FAIL (config) - {e}")
        return False
    except LLMUnavailable as e:
        print(f"  RESULT : FAIL (transient) - {e}")
        return False


print("--- LLM preflight ---")
check_llm()

## `ask()`

Runs the agent **and** the LLM, returns natural language. Never raises for a missing key
or a network blip — it degrades to the raw bundle and says so.

```python
ask("how many images are there?")      # aggregate -> survey
ask("where is safe_resolve defined?")  # locate -> grep
ask("what is in my practice project?") # bare keyword -> folder
ask("open config.yaml")                # no match -> near misses
```

Result contract (uniform across every agent, so `planner` can slot in unchanged):

| field | meaning |
|---|---|
| `status` | `ok` / `near_miss` / `not_found` |
| `route` | `aggregate` / `locate` |
| `summary` | one line, no dumps |
| `evidence` | citable excerpts |
| `confidence` | explicit, so the LLM hedges instead of inventing |

In [ ]:
# ===== ask() =====
NOT_FOUND_REPLY = ("I couldn't find anything resembling that. I searched by filename and "
                   "by file contents, and got no matches — so rather than guess, here's "
                   "what I looked for.")


def synthesize(result: dict):
    """Returns None on TRANSIENT failure; raises LLMConfigError on a CONFIG failure."""
    try:
        return call_llm(build_llm_messages(result))
    except LLMConfigError as e:
        log("synthesize", detail=str(e)[:90], status="config_error", tool="llm")
        raise
    except LLMUnavailable as e:
        log("synthesize", detail=str(e)[:90], status="error", tool="llm")
        return None


def ask(user_query: str, top_files: int = 3, root=None,
        exhaustive: bool | None = None) -> str | None:
    result = run_agent(user_query, top_files=top_files, root=root,
                       exhaustive=exhaustive)
    print("=" * 62)
    print(f"AGENT: {result['status']} | similar={len(result.get('similar_files', []))} "
          f"read={len(result['evidence'])} | {result['elapsed_ms']}ms")
    print("=" * 62)

    try:
        out = synthesize(result)
    except LLMConfigError as e:
        print("\n" + "!" * 62)
        print("!!! LLM MISCONFIGURED -- wiring bug, not a network blip")
        print("!" * 62)
        print(f"  {e}")
        st = llm_status()
        print(f"  key      : {'SET' if st['configured'] else 'NOT SET'} ({st['key_source']})")
        print(f"  model    : {st['model']}")
        print(f"  base_url : {st['base_url']}")
        print("\n  fix one of:")
        print(f'    export GROQ_API_KEY="gsk_..."      then Restart Kernel & Run All')
        print(f'    export LLM_MODEL="{GROQ_CHAT_MODELS[0]}"')
        print("\n  raw evidence follows:")
        print(build_prompt_bundle(result))
        return None

    if out:
        print("\n--- ANSWER ---\n")
        print(out["text"])
        print(f"\n[{out['model']} | {out['usage'].get('total_tokens', '?')} tokens | "
              f"conf {result['confidence']}]")
        return out["text"]

    if result["status"] == "not_found":
        print("\n--- ANSWER (offline) ---\n")
        print(NOT_FOUND_REPLY)
        return NOT_FOUND_REPLY

    print("\n--- LLM TEMPORARILY UNAVAILABLE: raw evidence instead ---")
    print(build_prompt_bundle(result))
    print("\n>> re-run check_llm() to confirm the service is back.")
    return None

## Run it

Edit a query and re-run. `show_logs()` was dropped from the pipeline; inspect `LOGS`
directly, or run `test_searchfiles.py` for the 28 regression tests.

In [ ]:
# ===== live run =====
answer = ask("count all images files in the system")

## Next agent

`planner` — **no tools at all**, reads only what it is handed. Its output schema becomes
the future executor's *input* schema, so define it as a typed, validated structure rather
than prose.

Two things to fix first if you want them: `node_modules` is not in `SKIP_DIRS` (it is
already covered by `SKIP_DIRS` as written — verify before changing), and Rust `target/`
holds 4857 of the 5130 visible files as build artifacts, which distorts every count.

---

# Part 2 — `planner`, orchestrator, verifier

The plan-and-execute loop. The planner thinks **once up front**, then plain Python
drives the steps:

```
task → PLANNER (DSPy + Pydantic) → Plan
          ↓
     ORCHESTRATOR (plain Python, no LLM)
          ↓  ready step → agent → result → VERIFIER
          ↓                              ├─ pass → next step
          │                              └─ fail → PATH LOG → replan (capped)
          ↓
     final answer
```

**The planner has no tools.** It never touches the filesystem — it only reads what it is
handed. That is enforced structurally below: nothing in the planner's signature accepts
a path, and the planner module is not given the search functions.

### Design rules being followed

- **The plan is structured data, not prose.** Validated by Pydantic, so a malformed plan
  raises instead of silently executing nonsense.
- **The orchestrator is code, not an LLM.** Predictable, cheap, and trivially debuggable.
  Independent steps are eligible to run in parallel.
- **Plans stay short** (3–7 steps). Long plans go stale the moment step 1 changes the picture.
- **Every step declares `expected_output`**, so the verifier has something to check against.
  That field is also the future executor's pass/fail test.
- **The planner sees summaries, not raw dumps.** The full raw log stays in `LOGS`.

### Live-verified constraints (measured, not assumed)

| finding | consequence |
|---|---|
| DSPy 3.4.0 + litellm **strips `openai/`** from `openai/gpt-oss-120b` and routes to `openai-chat` → **404** | must pass `provider="groq"` explicitly |
| Groq returns **404, not 401**, for an inaccessible model | config errors must be loud |
| The model emits `"depends_on": null` despite being told `[]` | plan JSON needs a repair pass before validation |
| The model invents agent names (`file_search_agent`) | agent names must be validated against a registry and remapped |
| `gpt-oss-*` spend 35–50 tokens on hidden reasoning | planner needs `max_tokens` headroom or it returns `""` |
| `ChainOfThought` adds a `reasoning` field per call | on a per-model daily limit, reasoning costs are real — enabled for the planner only |
| `difflib` rates `search` vs `src` at **0.667** (just `s`+`rc`) | fuzzy matching needs a minimum stem length, or instruction prose matches every `src/` on disk |
| Feeding the rubric raw JSON made it `fail` correct evidence | citations must be rendered `path:line` **first**, excerpts labelled partial |

## 1. Types

`Plan` and `Step` are Pydantic models, so a bad plan is a **validation error**, not a
silent bad execution. `AgentResult` and `Attempt` are the contract every agent must
return if it is to be verifiable — that is the interface the earlier discussion flagged as
the thing to nail down.

`Attempt` is the path-log record. The planner reads *only* the compact fields
(`agent`, `instruction`, `failure_type`, `failure_reason`) on replan, so the log stays
small in context while the full detail remains available for debugging.

In [ ]:
# ===== planner types =====
from dataclasses import dataclass, field, asdict
from enum import Enum
from typing import Any, Literal

from pydantic import BaseModel, Field, field_validator, model_validator


class StepStatus(str, Enum):
    pending = "pending"
    running = "running"
    done = "done"
    failed = "failed"
    inconclusive = "inconclusive"   # verifier said `unsure` -- NOT a failure
    skipped = "skipped"


class Verdict(str, Enum):
    pass_ = "pass"
    fail = "fail"
    unsure = "unsure"


class FailureType(str, Enum):
    none = "none"
    empty = "empty"              # agent returned nothing usable
    wrong = "wrong"              # returned something off-target
    error = "error"              # tool/LLM blew up
    timeout = "timeout"
    contradiction = "contradiction"   # conflicts with a verified earlier step
    repeated = "repeated"        # orchestrator rejected it as a retry of a dead end
    inconclusive = "inconclusive"   # verifier unsure: not wrong, just unproven


class Step(BaseModel):
    id: int
    agent: str
    instruction: str
    depends_on: list[int] = Field(default_factory=list)
    expected_output: str
    status: StepStatus = StepStatus.pending
    result: dict[str, Any] | None = None

    @field_validator("depends_on", mode="before")
    @classmethod
    def _null_deps(cls, v):
        # the model emits null for "no dependencies" despite being told []
        return [] if v is None else v


class Plan(BaseModel):
    goal: str
    steps: list[Step]

    @model_validator(mode="after")
    def _check_dag(self):
        ids = [s.id for s in self.steps]
        if len(ids) != len(set(ids)):
            raise ValueError(f"duplicate step ids: {ids}")
        for s in self.steps:
            for d in s.depends_on:
                if d not in ids:
                    raise ValueError(f"step {s.id} depends on unknown step {d}")
            if s.id in s.depends_on:
                raise ValueError(f"step {s.id} depends on itself")
        return self

    def by_id(self, i: int) -> Step | None:
        return next((s for s in self.steps if s.id == i), None)


@dataclass
class AgentResult:
    """What every agent must return to be verifiable. Uniform by construction."""
    status: Literal["ok", "near_miss", "not_found"]
    summary: str
    evidence: list[Any] = field(default_factory=list)
    confidence: float = 0.0
    meta: dict[str, Any] = field(default_factory=dict)


@dataclass
class Attempt:
    """One path-log entry. `compact()` is what the planner sees on replan."""
    attempt_id: int
    plan_version: int
    step_id: int
    agent: str
    instruction: str
    status: str
    result_summary: str
    verdict: Verdict
    failure_reason: str = ""
    failure_type: FailureType = FailureType.none
    evidence_checked: str = ""
    duration_ms: float | None = None

    def compact(self) -> str:
        if self.verdict is Verdict.pass_:
            return f"step {self.step_id} [{self.agent}] OK: {self.result_summary[:70]}"
        return (f"DEAD END step {self.step_id} [{self.agent}] "
                f"'{self.instruction[:60]}' -> {self.failure_type.value}: "
                f"{self.failure_reason[:70]}")

    def signature(self) -> tuple:
        """Used to detect the planner repeating a known dead end."""
        return (self.agent, re.sub(r"\s+", " ", self.instruction.strip().lower()))


PATH_LOG: list[Attempt] = []


def dead_ends() -> list[str]:
    """The compact 'do-not-repeat' list handed to the planner.

    Only genuine FAILURES belong here. An `inconclusive` attempt was never shown to be
    wrong -- including it would tell the planner "don't repeat this" about a step that
    simply could not be checked, which is how a planner learns to avoid correct work.
    """
    return [a.compact() for a in PATH_LOG if a.verdict is Verdict.fail]

## 2. Agent registry

The planner may only name an agent that exists here. `normalise_agent()` also repairs
the invented names models produce (`file_search_agent` → `searchFiles`), because
rejecting the whole plan over a naming quirk wastes a round trip — but an **unknown**
agent is a hard error, because that means the planner is hallucinating capability.

Only `searchFiles` exists today. `webSearch` is declared but unimplemented, and the
orchestrator will mark steps targeting it `skipped` rather than pretend.

In [ ]:
# ===== agent registry =====
AGENT_REGISTRY = {"searchFiles", "webSearch", "planner", "assistant"}
IMPLEMENTED_AGENTS = {"searchFiles"}

# model-invented aliases -> canonical name
# Keys are the agent name with every non-letter stripped, because normalise_agent()
# does exactly that before lookup. So "file_search_agent" -> "filesearchagent".
AGENT_ALIASES = {
    # searchFiles
    "filesearchagent": "searchFiles", "filesearch": "searchFiles",
    "search": "searchFiles", "searchfiles": "searchFiles",
    "searchfile": "searchFiles", "explore": "searchFiles", "finder": "searchFiles",
    "filesystem": "searchFiles", "locate": "searchFiles", "grep": "searchFiles",
    # webSearch
    "websearch": "webSearch", "web": "webSearch", "scout": "webSearch",
    "browse": "webSearch", "internet": "webSearch", "docs": "webSearch",
    # other roles
    "plan": "planner", "reason": "assistant", "assistant": "assistant",
}


def normalise_agent(name: str) -> str:
    """Map an alias to its canonical agent. Raises on genuinely unknown agents.

    Lookup key strips non-letters, so `file_search_agent`, `FileSearchAgent`, and
    `filesearch` all collapse to the same entry.
    """
    key = re.sub(r"[^a-z]", "", str(name).lower())
    canon = AGENT_ALIASES.get(key)
    if canon:
        return canon
    for a in AGENT_REGISTRY:
        if a.lower() == str(name).lower():
            return a
    raise ValueError(
        f"unknown agent {name!r}; available: {sorted(AGENT_REGISTRY)}")


def run_search_files(instruction: str, context: dict, top_files: int = 3) -> AgentResult:
    """Adapt the searchFiles agent to the AgentResult contract.

    `context` carries verified results from dependency steps. It is deliberately
    NOT concatenated into the query: the search agent does its own intent parsing and
    prepending prose would only pollute it.
    """
    t0 = time.perf_counter()
    res = run_agent(instruction, top_files=top_files)
    if res.get("route") == "aggregate":
        sv = res["survey"]
        return AgentResult(
            status="ok",
            summary=f"{sv['matched_count']} file(s) in {sv['by_category']} "
                    f"(of {sv['files_seen']} surveyed; excludes {len(sv['excluded_dirs'])} dirs)",
            evidence=[{"path": s, "kind": "sample"} for s in sv["sample"][:8]],
            confidence=res["confidence"],
            meta={"route": "aggregate", "counts": sv["by_category"],
                  "files_seen": sv["files_seen"], "excluded": sv["excluded_dirs"]})

    if res["status"] == "near_miss":
        return AgentResult(status="near_miss",
                           summary=res["summary"],
                           evidence=[{"path": str(c["path"]), "score": c["score"]}
                                     for c in res["near_misses"]],
                           confidence=res["confidence"],
                           meta={"route": "locate", "near_misses": res["near_misses"]})

    if res["status"] == "not_found":
        return AgentResult(status="not_found", summary=res["summary"],
                           evidence=[], confidence=0.0, meta={"route": "locate"})

    ev = [{"path": e["path"], "kind": e["kind"],
           "read_status": e["read_status"],
           "line": e.get("match_line"),
           "excerpt": e["excerpt"][:800]} for e in res["evidence"]]
    return AgentResult(status="ok", summary=res["summary"], evidence=ev,
                       confidence=res["confidence"],
                       meta={"route": "locate",
                             "similar": res["similar_files"][:10]})

AGENT_RUNNERS = {"searchFiles": run_search_files}

## 3. The planner (DSPy + Pydantic)

Two signatures: `MakePlan` for the initial plan, `RevisePlan` for replanning. Different
docstrings and different field descriptions — that is what makes them different modules to
DSPy, and it is why `RevisePlan` also sees the dead-ends list.

**`build_plan()` is the seam.** The body is one DSPy module call; swapping in
few-shot demos, or optimising with `GEPA`/`BootstrapFewShot`, happens inside it without
touching the orchestrator. That is the payoff of making the plan structured data.

### ChainOfThought — applied per module, deliberately uneven

`dspy.ChainOfThought(Sig)` adds a `reasoning` output field and wraps `Predict`
internally, so enabling it is a one-line swap. It is **not** right everywhere:

| module | CoT? | reasoning |
|---|---|---|
| `MakePlan` / `RevisePlan` | **yes** | choosing agents and step dependencies is multi-step logic; the rationale is also free debuggability |
| `VerifyResult` | **no** by default | a judge that reasons *before* answering tends to construct a justification for whichever verdict it lands on — the leniency risk this verifier was designed to avoid |
| `SolveWithTools` (ReAct) | n/a | ReAct already emits `thought_N` per iteration. A second reasoning channel would burn tokens for nothing. |

Override with `PLANNER_COT=0` / `VERIFY_COT=1`. CoT is not free: it adds tokens to every
call, which matters given Groq's per-model daily limit — and it makes `reasoning` visible
in the log either way, which is the part worth having.

`parse_plan_json()` exists because the model reliably emits three defects — `null`
dependencies, invented agent names, and prose wrapped around the JSON. All three are
repaired **before** validation, so a cosmetic slip does not cost a retry. Genuinely
malformed output still raises, and `make_plan` retries once.

In [ ]:
# ===== planner: DSPy signatures =====
import os
VENV_PY = "/home/princ3kr/Desktop/learnings/.venv/bin/python"

try:
    import dspy
    # `from dspy.signature import ...` does NOT work on dspy 3.x -- there is no
    # `dspy.signature` submodule. The names are re-exported at package top level.
    from dspy import Signature, InputField, OutputField
    DSPY_OK = True
except ImportError as _e:
    DSPY_OK = False
    _DSPY_IMPORT_ERROR = _e


def configure_dspy() -> bool:
    """Point DSPy at the same OpenAI-compatible endpoint as our own client.

    `provider="groq"` is REQUIRED: litellm otherwise strips `openai/` from
    `openai/gpt-oss-120b` and routes to the OpenAI endpoint, which 404s on Groq.
    """
    if not DSPY_OK:
        print(f"  dspy unavailable ({_DSPY_IMPORT_ERROR}) -- using make_plan_fallback")
        print(f"  install it with: {VENV_PY} -m pip install dspy")
        return False
    if not api_key():
        print("  no API key -> set GROQ_API_KEY")
        return False
    model = os.getenv("DSPY_MODEL", "openai/gpt-oss-120b")
    # num_retries=0: DSPy's internal retry storm turned ONE rubric call into 536
    # seconds of silent backoff when Groq returned 429. Fail fast instead and let
    # the orchestrator see the error and degrade visibly.
    dspy.configure(lm=dspy.LM(f"openai/{model}", provider="openai",
                              api_base=LLM_BASE_URL, api_key=api_key(),
                              temperature=0.0, max_tokens=2000, num_retries=0))
    print(f"  dspy configured: openai/{model} @ {LLM_BASE_URL}")
    return True


if DSPY_OK:
    class MakePlan(Signature):
        """Break a file-inspection task into 2-4 short ordered steps.

        Use only the `searchFiles` agent. One concern per step. Each step must be
        independently checkable, so give a concrete expected_output.
        Steps with no dependency must use an empty depends_on list.
        """
        task: str = InputField(desc="what the user wants to find out")
        findings_so_far: str = InputField(desc="verified results from earlier steps, or 'none'")
        steps_json: str = OutputField(
            desc=('JSON array, no prose. Each element: '
                  '{"id": 1, "agent": "searchFiles", "instruction": "...", '
                  '"depends_on": [], "expected_output": "..."}'))

    class RevisePlan(Signature):
        """Revise a stalled plan by changing strategy, not by rewording.

        You are given attempts that already failed. Do NOT repeat any of them:
        change the search strategy, use a different tool surface, widen or narrow
        the query, or split the step differently. Keep steps that already passed.
        Return only the REMAINING steps, renumbered from 1.
        """
        goal: str = InputField(desc="the original task")
        completed: str = InputField(desc="steps already verified, with their results")
        dead_ends: str = InputField(desc="attempts that FAILED - never repeat these")
        failure: str = InputField(desc="why the plan stalled")
        steps_json: str = OutputField(
            desc='JSON array of remaining steps only, no prose, same element shape')

    # ChainOfThought where reasoning genuinely helps the DECISION, plain Predict
    # where it would bias the judgement. See the markdown above for why these
    # are deliberately different.
    _cot_plan = os.getenv("PLANNER_COT", "1") not in ("0", "false", "no")

    _make = dspy.ChainOfThought(MakePlan) if _cot_plan else dspy.Predict(MakePlan)
    _revise = dspy.ChainOfThought(RevisePlan) if _cot_plan else dspy.Predict(RevisePlan)
    # _verify is built in the verifier cell, after VerifyResult exists
else:                                            # pragma: no cover
    _make = _revise = None


PLAN_INSTRUCTIONS = (
    "Rules: 2-4 steps. agent must be exactly 'searchFiles'. "
    "depends_on must be [] or a list of earlier step ids. "
    "One concern per step. expected_output must be concrete and checkable "
    "(e.g. 'list of file paths with line numbers'). "
    "Do not read file contents yourself - only the agents can."
)


def _extract_json(text: str):
    """Pull the first JSON array out of prose-wrapped output."""
    t = text.strip()
    if t.startswith("```"):
        t = re.sub(r"^```[a-zA-Z]*\n?", "", t)
        t = re.sub(r"\n?```$", "", t.strip())
    start = t.find("[")
    if start == -1:
        raise ValueError(f"no JSON array in output: {text[:120]!r}")
    depth, in_str, esc = 0, False, False
    for i in range(start, len(t)):
        ch = t[i]
        if in_str:
            if esc:
                esc = False
            elif ch == "\\":
                esc = True
            elif ch == '"':
                in_str = False
            continue
        if ch == '"':
            in_str = True
        elif ch in "[{":
            depth += 1
        elif ch in "]}":
            depth -= 1
            if depth == 0:
                return json.loads(t[start:i + 1])
    raise ValueError("unterminated JSON array")


def parse_plan_json(raw: str, goal: str) -> Plan:
    """Repair the three predictable defects, then validate.

    1. `depends_on: null`            -> []
    2. invented agent names         -> canonical name
    3. prose around the JSON        -> stripped
    """
    data = _extract_json(raw)
    cleaned = []
    for i, s in enumerate(data, 1):
        if not isinstance(s, dict):
            continue
        deps = s.get("depends_on") or []
        if isinstance(deps, (int, str)):
            deps = [deps]
        cleaned.append({
            "id": int(s.get("id", i)),
            "agent": normalise_agent(s.get("agent", "searchFiles")),
            "instruction": str(s.get("instruction", "")).strip(),
            "depends_on": [int(d) for d in deps],
            "expected_output": str(s.get("expected_output", "")).strip(),
        })
    cleaned = [s for s in cleaned if s["instruction"]]
    if not cleaned:
        raise ValueError("plan had no usable steps")
    cleaned = cleaned[:7]                      # long plans go stale; cap hard
    return Plan(goal=goal, steps=cleaned)


def make_plan(task: str, findings: str = "none") -> Plan:
    """THE SEAM: one Predict call. Swap in CoT / demos / an optimiser here."""
    if not DSPY_OK:
        raise RuntimeError("dspy unavailable")
    out = _make(task=task + "\n\n" + PLAN_INSTRUCTIONS,
                findings_so_far=findings)
    log("planner_reasoning", detail=str(getattr(out, "reasoning", ""))[:120],
        status="ok", cot=_cot_plan, tool="planner")
    return parse_plan_json(out.steps_json, goal=task)


def replan(goal: str, completed: str, failure: str,
           max_replans: int = 2) -> Plan:
    """Revise only the remaining steps. `dead_ends()` is the do-not-repeat list."""
    if not DSPY_OK:
        raise RuntimeError("dspy unavailable")
    out = _revise(goal=goal + "\n\n" + PLAN_INSTRUCTIONS,
                  completed=completed, dead_ends="; ".join(dead_ends()) or "none",
                  failure=failure)
    return parse_plan_json(out.steps_json, goal=goal)


def make_plan_fallback(task: str) -> Plan:
    """Deterministic plan, used when dspy is unavailable or keeps failing.

    Not a stub: it encodes the shape a good plan takes for this agent, so the
    orchestrator and verifier stay testable without a network call.
    """
    s1 = Step(id=1, agent="searchFiles", instruction=task,
              depends_on=[], expected_output="list of file paths with line numbers")
    s2 = Step(id=2, agent="searchFiles",
              instruction=f"file type breakdown relevant to: {task[:80]}",
              depends_on=[1], expected_output="counts per file extension")
    return Plan(goal=task, steps=[s1, s2])

## 4. Verifier

**Cheap deterministic checks first, LLM rubric only if those pass.** A rubric call on
obviously-empty output wastes a round trip and invites the model to talk itself into a
pass.

```python
verdict, reason = deterministic_verify(step, result)
if verdict is Verdict.unsure:                      # empty/short result only
    verdict, reason = rubric_verify(step, result)  # separate signature, own prompt
```

`unsure` is a real third outcome, not a cop-out: it means "one more cheap check, or ask
the user", and it is what stops the orchestrator burning replans on a step that was fine.

**Self-grading is deliberately avoided.** `VerifyResult` has a different docstring and
different field descriptions from the agents' prompts, so DSPy treats it as a separate
module. Same-signature self-grading is biased toward `pass`.

**`use_llm=False` still completes a plan.** A result that cleared every structural check
is accepted as `pass` but logged as `structural checks passed (rubric skipped)`. Returning
`unsure` there instead would mean *no step could ever be marked done*, so an offline run
could never finish. The weaker guarantee is labelled in the log rather than implied.

In [ ]:
# ===== verifier =====
def deterministic_verify(step: Step, result: AgentResult) -> tuple[Verdict, str]:
    """Structural checks. No LLM. Returns (Verdict, reason)."""
    if result is None:
        return Verdict.fail, "agent returned nothing"
    if result.status == "not_found":
        return Verdict.fail, "no files matched"
    if result.status == "near_miss":
        return Verdict.unsure, "only near matches, no exact hit"
    if not result.evidence:
        return Verdict.unsure, "ok but zero evidence items"
    if result.confidence <= 0.0:
        return Verdict.unsure, "zero confidence"

    # every cited path must exist and be inside the jail
    bad = []
    for e in result.evidence:
        p = e.get("path") if isinstance(e, dict) else None
        if not p:
            continue
        try:
            rp = safe_resolve(p)
            if not rp.exists():
                bad.append(f"{rp.name} missing")
        except PermissionError:
            bad.append(f"{p} outside jail")
    if bad:
        return Verdict.fail, f"unverifiable paths: {'; '.join(bad[:3])}"

    # expected_output mentions file paths -> require at least one that exists
    if "path" in step.expected_output.lower() and not any(
            isinstance(e, dict) and e.get("path") for e in result.evidence):
        return Verdict.fail, "expected file paths, got none"

    return Verdict.unsure, "structural checks passed; needs rubric"


if DSPY_OK:
    class VerifyResult(Signature):
        """Judge whether a result satisfies a step's requirement.

        The `citable locations found:` lines are authoritative -- they are paths and
        line numbers that really exist. Excerpts are partial windows and may show
        prose, imports, or surrounding text; do NOT fail a step merely because an
        excerpt does not show the exact construct. Judge whether the LOCATIONS could
        satisfy the requirement. Fail only when the locations clearly cannot.
        """
        requirement: str = InputField(desc="what the step expected")
        instruction: str = InputField(desc="what the agent was asked to do")
        result: str = InputField(desc="agent summary plus citable locations and excerpts")
        verdict: str = OutputField(desc="exactly one of: pass, fail, unsure")
        reason: str = OutputField(desc="one short sentence justifying the verdict")

    # CoT OFF by default for the judge: reasoning before a verdict gives the model
    # room to construct a justification for whatever it lands on. Flip with
    # VERIFY_COT=1 if you want to see the difference.
    _cot_verify = os.getenv("VERIFY_COT", "0") not in ("0", "false", "no")
    _verify = dspy.ChainOfThought(VerifyResult) if _cot_verify \
        else dspy.Predict(VerifyResult)


def _format_evidence(result: AgentResult, limit: int = 6) -> str:
    """Render evidence as citable path:line lines FIRST, excerpts after.

    Feeding raw JSON to the rubric was a real failure: a correct hit at
    `glinerTest.ipynb:15` was buried inside a markdown excerpt, the rubric saw prose
    instead of code, and it returned a confident `fail` on evidence that DID answer the
    step. The citations must lead, and excerpts must be labelled as partial.
    """
    cites, excerpts = [], []
    for e in result.evidence[:limit]:
        if not isinstance(e, dict):
            continue
        p = e.get("path", "?")
        line = e.get("line")
        cites.append(f"  {p}:{line}" if line else f"  {p}")
        ex = (e.get("excerpt") or "").strip().replace("\n", " ")[:220]
        if ex:
            excerpts.append(f"  [{p}] {ex}")
    out = ["citable locations found:"] + (cites or ["  (none)"])
    if excerpts:
        out += ["excerpts (PARTIAL - the location lines above are authoritative):"] \
               + excerpts
    return "\n".join(out)


def rubric_verify(step: Step, result: AgentResult, use_llm: bool = True) -> tuple[Verdict, str]:
    """LLM judgement, only for results that passed the structural checks."""
    if not use_llm or not DSPY_OK:
        return Verdict.unsure, "no rubric available; deferring to orchestrator"

    try:
        out = _verify(requirement=step.expected_output, instruction=step.instruction,
                      result=f"agent summary: {result.summary}\n"
                             f"{_format_evidence(result)}")
    except Exception as e:
        # The verifier must NEVER crash the orchestrator. A rate limit or transport
        # fault downgrades to `unsure`, which the orchestrator reports honestly
        # rather than treating as pass or fail.
        log("verify", detail=f"{type(e).__name__}: {str(e)[:70]}", status="error",
            tool="verifier")
        return Verdict.unsure, f"rubric unavailable ({type(e).__name__})"
    v = str(out.verdict).strip().lower()
    verdict = {"pass": Verdict.pass_, "fail": Verdict.fail,
               "unsure": Verdict.unsure}.get(v, Verdict.unsure)
    reasoning = str(getattr(out, "reasoning", "") or "")
    if reasoning:
        # keeping the rubric's reasoning is the cheapest way to audit WHY a step
        # was accepted -- these are the decisions that are hardest to review later
        log("rubric_reasoning", detail=reasoning[:120], status="ok",
            cot=_cot_verify, tool="verifier")
    return verdict, str(out.reason).strip()[:200]


def verify_step(step: Step, result: AgentResult, use_llm: bool = True) -> tuple[Verdict, str]:
    """Structural checks, then the rubric only if they pass. NEVER raises.

    A verifier that crashes the orchestrator is worse than useless, so any internal
    fault degrades to `unsure`, which the orchestrator reports honestly rather than
    silently counting as pass or fail.

    If `use_llm=False` there is no rubric, and returning `unsure` would mean NO step
    could ever be marked done -- the plan could not complete offline at all. So a
    result clearing every structural check is accepted as `pass`, explicitly labelled
    `rubric skipped` so the weaker guarantee is visible in the log, not implied.
    """
    try:
        v, why = deterministic_verify(step, result)
        if v is Verdict.fail:
            return v, why
        if v is Verdict.unsure and why == "structural checks passed; needs rubric":
            if not use_llm:
                return Verdict.pass_, "structural checks passed (rubric skipped)"
            return rubric_verify(step, result, use_llm=True)
        return v, why
    except Exception as e:
        log("verify", detail=f"{type(e).__name__}: {str(e)[:70]}", status="error",
            tool="verifier")
        return Verdict.unsure, f"verifier error ({type(e).__name__})"

## 5. Orchestrator — plain Python, no LLM

This is deliberate. An LLM orchestrator is harder to debug, costs a call per step, and
can loop forever. This one is a topological walk with three guards:

1. **Ready-queue** — only steps whose dependencies are `done` are eligible. Independent
   steps are collected together, so a future executor can run them in parallel; right now
   they run sequentially because `run_agent` shells out and parallelism buys little.
2. **Repeat rejection in code** — if a step matches a logged dead end
   (`(agent, instruction)`), it is marked `failed`/`repeated` and *not* executed. The
   planner cannot talk the loop into re-running a known dead end.
3. **Replan cap** — after `MAX_REPLANS`, it stops and reports honestly what worked, what
   didn't, and what was tried. Replanning only triggers on a genuine stall: if steps are
   still ready and nothing failed, the loop keeps executing instead of replanning. (It
   used to replan eagerly, which burned the budget planning work it could just do.)

Replanning **only touches pending steps**. Verified results are pinned, so a late replan
can't discard work that already passed.

### `unsure` is a status, not a failure

The verifier can return three verdicts, and the orchestrator gives `unsure` its own
status: `inconclusive`. Collapsing it into `failed` was wrong — when Groq rate-limited
the rubric, a perfectly good result was logged as `fail_type=wrong`, which claims the
agent returned something off-target when the truth is nobody could check. The run record
now separates `done` / `failed` / `inconclusive` / `skipped`, and the final answer states
which is which.

In [ ]:
# ===== orchestrator =====
MAX_REPLANS = 2
MAX_STEPS_PER_PLAN = 7


def ready_steps(plan: Plan) -> list[Step]:
    """Steps whose dependencies are all done."""
    done = {s.id for s in plan.steps if s.status is StepStatus.done}
    return [s for s in plan.steps
            if s.status is StepStatus.pending and all(d in done for d in s.depends_on)]


def blocked_steps(plan: Plan) -> list[Step]:
    """Pending steps that can never run because a dependency failed."""
    failed = {s.id for s in plan.steps if s.status is StepStatus.failed}
    return [s for s in plan.steps
            if s.status is StepStatus.pending
            and any(d in failed for d in s.depends_on)]


def _log_attempt(plan_version: int, step: Step, result: AgentResult | None,
                 verdict: Verdict, reason: str, duration_ms: float | None = None):
    a = Attempt(attempt_id=len(PATH_LOG) + 1, plan_version=plan_version,
                step_id=step.id, agent=step.agent, instruction=step.instruction,
                status=(result.status if result else "none"),
                result_summary=(result.summary if result else ""),
                verdict=verdict, failure_reason=reason,
                failure_type=_classify(result, verdict, reason),
                evidence_checked=(f"{len(result.evidence)} item(s)" if result else ""),
                duration_ms=duration_ms)
    PATH_LOG.append(a)
    log("plan_step", detail=f"v{plan_version} step {step.id} [{step.agent}] "
                            f"{verdict.value}: {reason[:50]}",
        status=verdict.value, step_id=step.id, agent=step.agent,
        attempt_id=a.attempt_id, duration_ms=duration_ms, tool="orchestrator")
    return a


def _classify(result: AgentResult | None, verdict: Verdict, reason: str) -> FailureType:
    if verdict is Verdict.pass_:
        return FailureType.none
    if verdict is Verdict.unsure:
        # `unsure` means unproven, not wrong. Classifying it as FailureType.wrong
        # made the path log claim the agent returned something off-target when in
        # fact the verifier simply could not reach a verdict.
        return FailureType.inconclusive
    if "outside jail" in reason or "missing" in reason:
        return FailureType.wrong
    if "timeout" in reason.lower():
        return FailureType.timeout
    if result is None:
        return FailureType.error
    if result.status == "not_found":
        return FailureType.empty
    if not result.evidence:
        return FailureType.empty
    return FailureType.wrong


def _verified_digest(plan: Plan) -> str:
    """Compact record of what has PASSED, so a replan can reuse it."""
    lines = []
    for s in plan.steps:
        if s.status is StepStatus.done and s.result:
            lines.append(f"step {s.id}: {s.result.get('summary', '')[:120]}")
    return "; ".join(lines) or "none"


def execute_plan(plan: Plan, plan_version: int = 1, use_llm_verify: bool = True,
                 replans_left: int = MAX_REPLANS) -> dict:
    """Run the plan to completion, replanning on failure. Returns a run record."""
    seen_sigs = {a.signature() for a in PATH_LOG}

    while True:
        for s in blocked_steps(plan):
            s.status = StepStatus.skipped
            _log_attempt(plan_version, s, None, Verdict.fail,
                         "dependency failed", None)

        ready = ready_steps(plan)
        if not ready:
            break

        for step in ready:
            sig = (step.agent, re.sub(r"\s+", " ", step.instruction.strip().lower()))
            if sig in seen_sigs:
                step.status = StepStatus.failed
                _log_attempt(plan_version, step, None, Verdict.fail,
                             "matches a logged dead end", None)
                continue

            if step.agent not in IMPLEMENTED_AGENTS:
                step.status = StepStatus.failed
                _log_attempt(plan_version, step, None, Verdict.fail,
                             f"agent {step.agent} not implemented", None)
                continue

            step.status = StepStatus.running
            # by_id() can legitimately return None if a replan ever leaves a
            # dangling dep, so resolve defensively rather than crashing the run
            ctx = {d.id: (d.result or {})
                   for d in (plan.by_id(x) for x in step.depends_on)
                   if d is not None}
            t0 = time.perf_counter()
            try:
                result = AGENT_RUNNERS[step.agent](step.instruction, ctx)
                err = None
            except Exception as e:
                result, err = None, f"{type(e).__name__}: {e}"
            dt = round((time.perf_counter() - t0) * 1000, 1)

            if err:
                step.status = StepStatus.failed
                _log_attempt(plan_version, step, None, Verdict.fail, err, dt)
                seen_sigs.add(sig)
                continue

            verdict, reason = verify_step(step, result, use_llm=use_llm_verify)
            step.result = asdict(result)
            if verdict is Verdict.pass_:
                step.status = StepStatus.done
                _log_attempt(plan_version, step, result, verdict, reason, dt)
            elif verdict is Verdict.unsure:
                # `unsure` is NOT a failure. Treating it as one made a rate-limited
                # rubric look like a broken step, which is a different claim and a
                # wrong one. It gets its own status and is reported separately.
                step.status = StepStatus.inconclusive
                _log_attempt(plan_version, step, result, verdict, reason, dt)
            else:
                step.status = StepStatus.failed
                _log_attempt(plan_version, step, result, verdict, reason, dt)
                seen_sigs.add(sig)

        if all(s.status is not StepStatus.pending for s in plan.steps):
            break
        if not ready_steps(plan):
            break                     # stalled: only blocked steps remain
        if not any(s.status is StepStatus.failed for s in plan.steps):
            continue                  # nothing failed -- just keep executing
        if replans_left <= 0:
            break

        failed = [s for s in plan.steps if s.status is StepStatus.failed]
        pinned = [s for s in plan.steps if s.status is StepStatus.done]
        failure = "; ".join(f"step {s.id}: {s.result.get('summary','')[:60] if s.result else ''}"
                            for s in failed) or "unknown"
        try:
            new_plan = replan(plan.goal, _verified_digest(plan), failure,
                              max_replans=replans_left)
        except Exception as e:
            _log_attempt(plan_version, Step(id=-1, agent="planner",
                                           instruction="replan", expected_output=""),
                         None, Verdict.fail,
                         f"replan failed: {type(e).__name__}: {str(e)[:70]}", None)
            break

        # Keep verified steps pinned at their ORIGINAL ids, and remap the new
        # remainder through an explicit old->new map. Shifting every new id by a
        # flat offset is wrong: the planner's new deps refer to the ORIGINAL ids,
        # and pinned steps keep theirs, so a blind offset left deps pointing at ids
        # that no longer exist.
        offset = max((s.id for s in pinned), default=0)
        id_map = {s.id: s.id for s in pinned}
        for n, s in enumerate(new_plan.steps, 1):
            id_map[s.id] = offset + n
        for n, s in enumerate(new_plan.steps, 1):
            s.id = offset + n
            s.depends_on = sorted({id_map[d] for d in s.depends_on if d in id_map})
        plan.steps = pinned + new_plan.steps
        # a dep that could not be remapped would silently never run
        live_ids = {s.id for s in plan.steps}
        for s in plan.steps:
            if any(d not in live_ids for d in s.depends_on):
                s.depends_on = [d for d in s.depends_on if d in live_ids]
        plan_version += 1
        replans_left -= 1
        _log_attempt(plan_version,
                     Step(id=-1, agent="planner", instruction="replan",
                          expected_output="revised remaining steps"),
                     AgentResult(status="ok",
                                 summary=f"plan v{plan_version} with "
                                         f"{len(new_plan.steps)} new step(s)",
                                 confidence=1.0),
                     Verdict.pass_, "replanned", None)

    return {"plan": plan, "version": plan_version,
            "done": [s.id for s in plan.steps if s.status is StepStatus.done],
            "failed": [s.id for s in plan.steps if s.status is StepStatus.failed],
            "inconclusive": [s.id for s in plan.steps
                             if s.status is StepStatus.inconclusive],
            "skipped": [s.id for s in plan.steps if s.status is StepStatus.skipped],
            "attempts": len(PATH_LOG)}

## 6. Run it

`run_task()` decides whether a plan is needed at all — a simple request is answered
directly by `searchFiles`, no planner, no orchestrator. That gate is a plain rule for now
and is logged, so it can be tuned (or handed to DSPy) later from real data.

In [ ]:
# ===== entry point =====
import threading

PLAN_NEEDED_CUES = ("then", "and also", "after that", "compare", "compare them",
                    "both", "across", "summarize", "summarise", "then find",
                    "and count", "and report", "step by step")


def needs_plan(query: str) -> tuple[bool, str]:
    """Cheap gate. Logged so the threshold can be tuned from data later."""
    q = query.lower()
    cue = next((c for c in PLAN_NEEDED_CUES if c in q), None)
    multi = len(re.findall(r"[a-z_]+\.(?:py|md|ipynb|json|yaml|txt)", q)) >= 2
    decision = bool(cue) or multi
    reason = f"cue={cue!r}" if cue else ("multiple files named" if multi else "single-step")
    log("gate", detail=f"needs_plan={decision} ({reason})", status="ok",
        decision=decision, reason=reason, tool="router")
    return decision, reason


def run_task(query: str, strategy: str = "auto", use_llm_verify: bool = True,
             replans_left: int = MAX_REPLANS) -> str | None:
    """Main entry.

    strategy:
      auto   -- single-step -> direct ask(); otherwise plan-execute (deterministic)
      direct -- one searchFiles call, no planner
      plan   -- planner + orchestrator + verifier
      react  -- dspy.ReAct adaptive loop
    """
    if strategy == "auto":
        use_planner, _ = needs_plan(query)
        strategy = "plan" if use_planner else "direct"
        log("strategy", detail=detail_of(strategy), status="ok",
            strategy=strategy, tool="router")
        print(f"  strategy: {strategy}")

    if strategy == "direct":
        return ask(query)

    if strategy == "react":
        return run_react(query)

    if strategy != "plan":
        raise ValueError(f"unknown strategy {strategy!r}")

    use_planner = True

    if not use_planner:
        return ask(query)

    t0 = time.perf_counter()

    # DSPy is configured lazily, and ONLY when a plan is actually needed. Without
    # this the fallback path calls rubric_verify against an unconfigured LM and
    # raises "No LM is loaded" instead of degrading.
    used_fallback = False
    plan = None
    if DSPY_OK:
        configure_dspy()
        try:
            plan = make_plan(query)
            log("plan", detail=f"v1 {len(plan.steps)} steps", status="ok",
                steps=len(plan.steps), tool="planner")
        except Exception as e:
            log("plan", detail=f"{type(e).__name__}: {str(e)[:60]}",
                status="fallback", tool="planner")
            plan = None
    if plan is None:
        used_fallback = True
        plan = make_plan_fallback(query)
        print("  (using the deterministic fallback plan -- DSPy plan unavailable)")

    print(f"\nPLAN (goal: {plan.goal[:70]})")
    for s in plan.steps:
        print(f"  {s.id}. [{s.agent}] {s.instruction[:64]}")
        print(f"     expects: {s.expected_output[:64]}  deps={s.depends_on}")

    run = execute_plan(plan, use_llm_verify=use_llm_verify, replans_left=replans_left)

    print(f"\nRUN: version={run['version']} done={run['done']} "
          f"failed={run['failed']} inconclusive={run['inconclusive']} "
          f"skipped={run['skipped']} "
          f"({round((time.perf_counter()-t0)*1000)}ms, {run['attempts']} attempts)")

    # If we fell back to a deterministic plan, the LM was never configured, so any
    # rubric call inside execute_plan would have raised. Re-run without it.
    if used_fallback and use_llm_verify:
        print("  (re-running with structural-only verification: no LM configured)")
        for s in plan.steps:
            s.status = StepStatus.pending
            s.result = None
        PATH_LOG.clear()
        run = execute_plan(plan, use_llm_verify=False, replans_left=0)

    if run["done"]:
        # verified results only. Failed/inconclusive steps contribute context,
        # never claims.
        ctx = "\n\n".join(
            f"STEP {s.id} (verified): {s.result['summary']}\n"
            + "\n".join(f"  - {e.get('path')}" for e in s.result["evidence"][:6]
                        if isinstance(e, dict) and e.get("path"))
            for s in run["plan"].steps if s.status is StepStatus.done and s.result)
        gaps = []
        for s in run["plan"].steps:
            if s.status is StepStatus.failed:
                gaps.append(f"step {s.id} [{s.agent}] FAILED")
            elif s.status is StepStatus.inconclusive:
                gaps.append(f"step {s.id} [{s.agent}] UNVERIFIED (could not be checked)")
            elif s.status is StepStatus.skipped:
                gaps.append(f"step {s.id} [{s.agent}] SKIPPED (dependency failed)")
        if gaps:
            ctx += "\n\nUNRESOLVED: " + "; ".join(gaps)

        messages = [
            {"role": "system", "content":
             "You are composing the final answer to a multi-step file-search task. "
             "Use ONLY the VERIFIED results below. Cite files as path:line. "
             "If some steps failed, say plainly which parts are unanswered and why. "
             "Never present an unverified result as fact. Be concise."},
            {"role": "user", "content":
             f"TASK: {query}\n\n{ctx}\n\n"
             f"DEAD ENDS TRIED ({len(dead_ends())}): {'; '.join(dead_ends()) or 'none'}"},
        ]
        try:
            out = call_llm(messages)
            print("\n--- FINAL ANSWER ---\n")
            print(out["text"])
            print(f"\n[{out['model']} | {out['usage'].get('total_tokens','?')} tokens]")
            return out["text"]
        except (LLMUnavailable, LLMConfigError) as e:
            log("final", detail=str(e)[:60], status="error", tool="llm")
            print(f"\nLLM unavailable: {e}\n\nVERIFIED RESULTS:\n{ctx}")
            return None
    else:
        print("\nNo step verified. Reporting honestly rather than guessing.")
        for a in PATH_LOG[-6:]:
            print(f"  attempt {a.attempt_id}: {a.compact()}")
        return None

## 8. ReAct — the adaptive alternative

The planner path is **deterministic**: it fixes the steps up front and executes them. That
is debuggable and cheap, and correct when you already know the shape of the work. But it
cannot react to something it did not anticipate — if step 2 returns an unexpected shape,
the plan is already committed.

`dspy.ReAct` closes that gap. The model interleaves **thought → tool call → observation**
until it chooses `finish`, choosing tools itself:

```
question → ReAct loop (max_iters hard cap)
             ├─ thought_0, find_files("glinerTest.ipynb")
             ├─ observation_0: "exact: /home/.../glinerTest.ipynb"
             ├─ thought_1, search_content("def safe_resolve")
             ├─ observation_1: ".../glinerTest.ipynb:151"
             └─ finish → answer
```

### When to use which

| | plan-execute | ReAct |
|---|---|---|
| steps known in advance | ✅ | wasteful — re-derives them |
| unknown shape, needs probing | brittle | ✅ |
| reproducibility | same query → same plan | varies run to run |
| cost | one plan + one call per step | a call **per iteration** |
| debuggability | step boundaries explicit | trajectory must be read |

Both are kept because they fail differently. `run_task(strategy=...)` picks.

**Safety is unchanged.** The ReAct tools are thin wrappers over the same
`find_files` / `search_content` / `read_file` / `survey_files`, so every call still goes
through `safe_resolve` and the same jail. The model chooses *which* tool to call; it
cannot reach anything the wrappers do not expose. `max_iters` is a hard cap, and tool
errors become observations rather than exceptions, so a bad call cannot kill the loop.

**Live-verified on this machine:** ReAct answered *"where is safe_resolve defined"* in
**9.1s over 3 iterations** — `search_content("def safe_resolve")` → `read_file(line 140)` →
`finish` — and cited `glinerTest.ipynb:143`. It chose the tools itself, with no plan.

### The Groq daily limit is PER MODEL

Hit while testing: `gpt-oss-120b` returned
`Rate limit ... on tokens per day (TPD): Limit 200000, Used 199251`. **`qwen/qwen3.8-27b`
worked immediately on the same key.** The bucket is per-model, so one model exhausting its
daily allowance does not lock out the others.

```bash
export DSPY_MODEL="qwen/qwen3.8-27b"   # ~4x faster, no reasoning tokens
```

This also sets the default ceiling honestly: a ReAct loop costs a call *per iteration*,
so a runaway loop on a tight limit is expensive. `max_iters=6` is deliberate.

In [ ]:
# ===== ReAct: adaptive tool loop =====
if DSPY_OK:
    from dspy.adapters.types.tool import Tool

    class SolveWithTools(Signature):
        """Answer questions about files on this computer using the supplied tools.

        Rules:
        - Call a tool rather than guessing. Never invent a path or a line number.
        - Cite sources as path:line in your answer.
        - If the tools cannot find something, say so plainly instead of speculating.
        - When you have enough, call `finish`.
        """
        question: str = InputField(desc="what the user wants to know")
        answer: str = OutputField(desc="the natural-language answer")

    # --- tool wrappers. Thin: they route to the SAME jailed functions, so the
    #     model picks the tool but cannot escape the jail or the allowlist.
    def _t_find(pattern: str) -> str:
        """Find files and folders whose name resembles `pattern`.

        Use for 'where is X', 'find X', or when you know part of a filename.
        Returns ranked matches: exact, glob, substring, or fuzzy.
        """
        hits = call_tool("find", pattern=pattern, max_results=8)
        REACT_CALLS.append(("find_files", {"pattern": pattern}, len(hits)))
        if not hits:
            return "no filename matched"
        return "\n".join(f"[{h['kind']}] {h['path']}" for h in hits[:8])

    def _t_grep(query: str) -> str:
        """Search inside file CONTENTS for a literal string.

        Use when you know a symbol, function name, or phrase but not the file.
        Returns path:line: matched-text.
        """
        hits = call_tool("grep", query=query, max_results=8)
        REACT_CALLS.append(("search_content", {"query": query}, len(hits)))
        if not hits:
            return "no content matched"
        return "\n".join(f"{h['path']}:{h['line_no']}: {h['text'][:80]}" for h in hits[:8])

    def _t_read(path: str, start_line: int = 1) -> str:
        """Read a capped window of a text file, starting at a line number.

        Use only after find or grep gave you a path. Archives are listed, not extracted.
        """
        r = call_tool("read", path=path, start_line=int(start_line))
        REACT_CALLS.append(("read_file", {"path": path}, 1 if r["status"] == "ok" else 0))
        if r["status"] != "ok":
            return f"could not read ({r['status']}): {r.get('reason', '')}"
        return r["text"][:2500]

    def _t_survey(category: str = "code") -> str:
        """Count files by category across the tree: code, image, video, audio,
        document, spreadsheet, notebook, archive, data, binary.

        Use for 'how many X'. Returns counts, and the directories that were excluded.
        """
        cat = (category or "code").strip().lower()
        sv = call_tool("survey", categories=[cat])
        REACT_CALLS.append(("survey_files", {"category": cat}, sv["matched_count"]))
        lines = [f"{c}: {n}" for c, n in sv["by_category"].items()]
        lines.append(f"total files visible: {sv['files_seen']}")
        lines.append(f"EXCLUDES: {', '.join(sv['excluded_dirs'][:8])} ...")
        return "\n".join(lines)

    REACT_TOOLS = [
        Tool(_t_find, name="find_files", desc=_t_find.__doc__.strip()),
        Tool(_t_grep, name="search_content", desc=_t_grep.__doc__.strip()),
        Tool(_t_read, name="read_file", desc=_t_read.__doc__.strip()),
        Tool(_t_survey, name="survey_files", desc=_t_survey.__doc__.strip()),
    ]
    _react = dspy.ReAct(SolveWithTools, tools=REACT_TOOLS, max_iters=6)
else:
    REACT_TOOLS = []
    _react = None

REACT_CALLS: list[tuple] = []


def render_trajectory(traj: dict) -> str:
    """ReAct returns a dict of thought_N / tool_name_N / tool_args_N / observation_N."""
    out = []
    i = 0
    while f"thought_{i}" in traj:
        out.append(f"[{i}] thought : {str(traj[f'thought_{i}'])[:150]}")
        out.append(f"    tool   : {traj.get(f'tool_name_{i}')}"
                   f"({traj.get(f'tool_args_{i}')})")
        out.append(f"    observe: {str(traj.get(f'observation_{i}', ''))[:220]}")
        i += 1
    return "\n".join(out) or "(empty trajectory)"


def run_react(query: str, max_iters: int = 6) -> str | None:
    """ReAct loop, logged into PATH_LOG in the same shape as a planned step."""
    if not DSPY_OK or _react is None:
        print("  dspy unavailable -- ReAct disabled")
        return None
    configure_dspy()
    REACT_CALLS.clear()
    t0 = time.perf_counter()
    try:
        pred = _react(question=query)
    except Exception as e:
        log("react", detail=f"{type(e).__name__}: {str(e)[:60]}", status="error",
            tool="react")
        print(f"  ReAct failed: {type(e).__name__}: {str(e)[:120]}")
        return None

    traj = dict(getattr(pred, "trajectory", {}) or {})
    answer = str(getattr(pred, "answer", "") or "")
    dt = round((time.perf_counter() - t0) * 1000)

    iterations = sum(1 for k in traj if k.startswith("thought_"))
    print(f"\nREACT: {len(REACT_CALLS)} tool call(s), {iterations} iteration(s), {dt}ms")
    print(render_trajectory(traj))

    # one PATH_LOG entry per tool call, so ReAct runs debug like planned ones
    for name, args, n in REACT_CALLS:
        ver = Verdict.pass_ if n else Verdict.fail
        _log_attempt(1, Step(id=-1, agent="react", instruction=f"{name}({args})",
                             expected_output="useful observations"),
                     AgentResult(status="ok" if n else "not_found",
                                 summary=f"{name} -> {n} hit(s)", confidence=0.8),
                     ver, "observed results" if n else "no hits", None)

    log("react", detail=f"{len(REACT_CALLS)} calls, {iterations} iters",
        status="ok", t0=None, calls=len(REACT_CALLS), iterations=iterations,
        tool="react")

    if answer:
        print("\n--- REACT ANSWER ---\n")
        print(answer)
    return answer or None

## 7. Guardrails and what's deliberately missing

**Caps:** `MAX_REPLANS=2`, `MAX_STEPS_PER_PLAN=7`. Replans are bounded so a planner
loop cannot run forever, and plan length is capped because long plans go stale.

**Repeat rejection is in code**, not in the prompt — the orchestrator compares
`(agent, instruction)` against every logged dead end and refuses to execute a match.

**Unimplemented agents are marked failed, not silently skipped.** A plan targeting
`webSearch` fails visibly rather than appearing to succeed.

### Practical constraint found while testing: Groq rate limits

The free tier rate-limits aggressively. Two consequences, both handled:

- **`num_retries=0` on the DSPy LM.** DSPy's internal retry storm turned *one* rubric
  call into **536 seconds** of silent backoff. Failing fast in 686ms and reporting the
  error is strictly better than a hang.
- **A rate-limited rubric becomes `inconclusive`, not `fail`.** Nothing was proven wrong,
  so the path log must not claim `failure_type=wrong`. The run then says
  `UNVERIFIED (could not be checked)` rather than asserting the agent failed.

If you hit this often: `DSPY_MODEL=qwen/qwen3.8-27b` is ~4× faster and burns no
reasoning tokens, or add an explicit backoff at the orchestrator level.

### Not built yet, and why it matters

- **`webSearch`** — declared in the registry, unimplemented. Needs the local
  `importlib`/`pip show` check *before* any network call, and fetched text must be
  treated as untrusted data. ReAct is where this would matter most: the model decides
  when to go to the network, so that check must live in the tool wrapper, not the prompt.
- **ReAct is not verified.** A planned step goes through `verify_step`; a ReAct run does
  not. Its final answer is whatever the model wrote from its own trajectory. The path log
  records the tool calls, but nothing checks the answer. Reusing `rubric_verify` on the
  trajectory is the obvious next step, and it is the main gap in having two strategies.
- **The verifier is the weak point.** It checks structure and asks one rubric question.
  It cannot tell you a file is *semantically* irrelevant — only a human or a much
  stronger check can. `unsure` exists so this shows up as ambiguity rather than a
  confident wrong answer.
- **TabPFN / DSPy optimisation.** `PATH_LOG` is now the training set. The columns that
  matter for "which steps are likely to fail" are already there: `agent`, `failure_type`,
  `depends_on`, `instruction` length.
- **A `SENSITIVE_NAMES` deny-list.** The jail spans your whole home, and
  `~/Downloads/AMRv2_api_key.txt` is currently readable by `searchFiles`.

### Next

A **sandboxed executor with explicit approval**, since `Plan`/`Step` already validate
and `execute_plan` already respects the DAG. Its input is exactly this `Step` object.

In [ ]:
# ===== introspection helpers =====
def show_path_log(last: int | None = None) -> None:
    """The path log, top to bottom -- how you debug any bad answer."""
    rows = PATH_LOG if last is None else PATH_LOG[-last:]
    if not rows:
        print("PATH_LOG is empty.")
        return
    print(f"{'#':>3} {'plan':>4} {'step':>4} {'agent':<13} {'verdict':<7} "
          f"{'fail_type':<13} detail")
    print("-" * 96)
    for a in rows:
        detail = a.result_summary or a.failure_reason
        print(f"{a.attempt_id:>3} v{a.plan_version:<3} {a.step_id:>4} "
              f"{a.agent:<13} {a.verdict.value:<7} {a.failure_type.value:<13} "
              f"{str(detail)[:44]}")
    print("-" * 96)
    ok = sum(1 for a in rows if a.verdict is Verdict.pass_)
    print(f"{len(rows)} attempt(s): {ok} pass, {len(rows)-ok} not")


def show_plan(plan: Plan) -> None:
    for s in plan.steps:
        mark = {StepStatus.done: "x", StepStatus.failed: "!",
                StepStatus.skipped: "-", StepStatus.running: ">"}.get(s.status, " ")
        print(f" [{mark}] {s.id}. [{s.agent}] {s.instruction[:56]} "
              f"deps={s.depends_on}")


print("planner ready:", "dspy OK" if DSPY_OK else "dspy MISSING (fallback plans)")
print("react   ready:", f"{len(REACT_TOOLS)} tools, max_iters=6"
      if _react is not None else "dspy MISSING (ReAct disabled)")
print("\nrun_task(query, strategy='auto'|'direct'|'plan'|'react')")

In [ ]:
run_task("how many pdf files are there in the system?", strategy='auto')

In [ ]:
PLAN_LOG